# Matura results: analysis and export

The new-formula matura (EM2023) for licea and technika, from CIE's school-level
files. The design is `specs/2026-10-05-matura-extension-design.md` in the workspace;
this notebook is the record of every figure it quotes. Sections:

0. Setup — paths, run flags, constants.
1. Load — the source files, the population rule, the rejected rows.
2. Subjects — the extended-subject coverage rule.
3. Why compare within type.
4. Choosing the metric — leave-one-out stability per type.
5. Final metric and colours.
6. Export — three population datasets and the analyst files.

## 0. Setup

In [1]:
# Run from repo/notebooks:  uv run jupyter lab
# Pure helpers live in ../src (unit-tested in tests/), on the path rather than
# installed, as in the E8 notebook. Each section imports the helpers it uses.
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('../src').resolve()))

DATA_DIR = Path('../data/egzamin-maturalny')
OUTPUT_DIR = Path('../output/matura')          # analyst xlsx and reports
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DATA_DIR = Path('../docs/data/matura')    # JSON for the map, one directory per population
DOCS_DATA_DIR.mkdir(parents=True, exist_ok=True)
GEO_DIR = Path('../docs/geo')                  # PRG polygons, shared with E8
COORDS_PATH = Path('../data/school_coords.csv')  # coordinate cache, shared with E8

# ── Run configuration (as in the E8 notebook) ────────────────────────────────
# FORCE_REGENERATE:   skip the idempotence check and always rewrite the exports.
# WRITE_OUTPUT_FILES: actually write files. Set False in an archived copy so
#   re-running it cannot overwrite the live data.
FORCE_REGENERATE = False
WRITE_OUTPUT_FILES = True

PRIMARY_REFERENCE_LEVEL = 'voivodeship'   # mean/median and the analyst xlsx use it
MATURA_LEVELS = ['national', 'voivodeship']  # spec §5.3: no powiat, no gmina
EXPORT_METRICS = ['mean', 'median', 'diff_mean', 'unit_norm_diff_mean']
# Hand-set, as in E8. Section 4's leave-one-out test is evidence, not a selector.
PRIMARY_METRIC = 'unit_norm_diff_mean'

## 1. Load

Each year is one file, pinned to its September edition (spec §2.2). The header is
three rows deep — band, subject, measure — and `apply_header` keys every column as
`(band, subject, measure)` (spec §2.3). Two checks run on every file before anything
is concatenated:

- every column the pipeline reads is present, so a column renamed in one year
  stops here instead of surviving as all-NaN for that year;
- no subject without a parenthesised suffix carries a published mean. The one
  such subject is maths sat in Lithuanian at one liceum in Puńsk, a different exam
  that must never merge into ordinary maths.

In [2]:
# ── Load the pinned September files ──────────────────────────────────────────
from school_quality.header import apply_header
from school_quality.matura import (
    BASIC_SUBJECTS,
    EXTENDED_SUBJECTS,
    MEASURE_MEAN,
    MEASURE_MEDIAN,
    MEASURE_N,
    MEASURE_PASS_RATE,
    WHOLE_EXAM,
    assert_unsuffixed_unpublished,
)

YEAR_FILE_RE = re.compile(r'^\d{4}')
META_LABELS = ['wojewodztwo - nazwa', 'powiat - nazwa', 'gmina - nazwa', 'typ gminy',
               'kod teryt gminy', 'rspo szkoly', 'nazwa szkoly', 'miejscowosc', 'ulica nr',
               'typ placowki', 'rodzaj placowki', 'czy publiczna']
REQUIRED = (
    [('meta', '', label) for label in META_LABELS]
    + [(*WHOLE_EXAM, MEASURE_PASS_RATE)]
    + [(band, subject, measure)
       for band, subject in [*BASIC_SUBJECTS.values(), *EXTENDED_SUBJECTS.values()]
       for measure in (MEASURE_N, MEASURE_MEAN, MEASURE_MEDIAN)]
)


def source_files(directory):
    return sorted(p for p in directory.glob('*.xlsx*') if not p.name.startswith('.'))


# Provenance: every source file must be documented in SOURCES.csv (as for E8).
sources = pd.read_csv(DATA_DIR / 'SOURCES.csv', dtype=str)
documented = {name.strip() for name in sources['file_name'].dropna()}
present = {p.name for p in source_files(DATA_DIR)}
undocumented = present - documented
if undocumented:
    raise ValueError(
        f'Source files in {DATA_DIR} missing from SOURCES.csv: {sorted(undocumented)}.\n'
        f'Add a row documenting where each came from.')
stale = documented - present
if stale:
    print(f'WARNING: SOURCES.csv lists files not on disk: {sorted(stale)}')

files = [p for p in source_files(DATA_DIR) if YEAR_FILE_RE.match(p.name)]
assert files, f'No data files found in {DATA_DIR}'

RAW = {}
for path in files:
    year = int(YEAR_FILE_RE.match(path.name).group(0))
    frame = apply_header(pd.read_excel(path, sheet_name='SAS', header=None))
    missing = [key for key in REQUIRED if key not in frame.columns]
    if missing:
        raise ValueError(f'{path.name}: required columns missing: {missing}')
    assert_unsuffixed_unpublished(frame.notna().sum().to_dict())
    RAW[year] = frame
    print(f'{path.name}: {len(frame):,} rows, {frame.shape[1]} columns')
print(f'Loaded {len(RAW)} files: {sorted(RAW)}; {sum(len(f) for f in RAW.values()):,} rows')

2023 - EM2023_2023_szkoly_09.xlsx: 2,322 rows, 424 columns


2024 - EM2023_2024_szkoly_09.xlsx: 4,717 rows, 423 columns


2025 - EM2023_2025_szkoly_09.xlsx: 4,797 rows, 431 columns


2026 - EM2023_2026_szkoly_09.xlsx: 4,875 rows, 464 columns
Loaded 4 files: [2023, 2024, 2025, 2026]; 16,711 rows


### The population rule (spec §3.1)

Rules 1–6 give a reject reason or the school's type (and raise on a category the
rule has never seen). A typed row is then rejected as `technikum_2023` (rule 7) or
as `missing_basic_subject` (rule 8), in that order. `nie dotyczy` licea are art
schools — licea sztuk plastycznych, music and ballet schools — and every one of
their names must say so.

In [3]:
# ── One row per (school, year), classified ───────────────────────────────────
from school_quality.levels import add_level_keys
from school_quality.matura import (
    BASIC_KEYS,
    REJECT_REASONS,
    assert_art_school_names,
    classify,
)
from school_quality.teryt import normalise_teryt

META_COLUMNS = {
    'school_name': 'nazwa szkoly', 'is_public': 'czy publiczna',
    'wojewodztwo': 'wojewodztwo - nazwa', 'powiat': 'powiat - nazwa',
    'gmina': 'gmina - nazwa', 'typ_gminy': 'typ gminy', 'miejscowosc': 'miejscowosc',
    'ulica_nr': 'ulica nr', 'teryt_raw': 'kod teryt gminy',
    'typ_placowki': 'typ placowki', 'rodzaj_placowki': 'rodzaj placowki',
}
SUBJECT_COLUMNS = {**BASIC_SUBJECTS, **EXTENDED_SUBJECTS}
TYPES = ('liceum', 'technikum')


def stripped(value):
    return value.strip() if isinstance(value, str) else value


year_frames = []
for year, frame in RAW.items():
    rows = pd.DataFrame({
        'rspo': pd.to_numeric(frame[('meta', '', 'rspo szkoly')], errors='raise').astype(int),
        'year': year,
    })
    for column, label in META_COLUMNS.items():
        rows[column] = frame[('meta', '', label)].map(stripped)
    rows['pass_rate'] = pd.to_numeric(frame[(*WHOLE_EXAM, MEASURE_PASS_RATE)], errors='raise')
    for key, (band, subject) in SUBJECT_COLUMNS.items():
        rows[f'n_{key}'] = pd.to_numeric(frame[(band, subject, MEASURE_N)], errors='raise')
        rows[f'mean_{key}'] = pd.to_numeric(frame[(band, subject, MEASURE_MEAN)], errors='raise')
        rows[f'median_{key}'] = pd.to_numeric(frame[(band, subject, MEASURE_MEDIAN)],
                                              errors='raise')
    has_basic = rows[[f'mean_{key}' for key in BASIC_KEYS]].notna().all(axis=1)
    rows['outcome'] = [classify(typ, rodzaj, year, bool(has))
                       for typ, rodzaj, has in zip(rows['typ_placowki'], rows['rodzaj_placowki'],
                                                   has_basic, strict=True)]
    year_frames.append(rows)
all_rows = pd.concat(year_frames, ignore_index=True)

outcome_table = (pd.crosstab(all_rows['outcome'], all_rows['year'])
                 .reindex(['liceum', 'technikum', *REJECT_REASONS], fill_value=0))
outcome_table['total'] = outcome_table.sum(axis=1)
outcome_table.loc['rows in file'] = outcome_table.sum()
print(outcome_table.to_string())

art = (all_rows['typ_placowki'] == 'liceum') & (all_rows['rodzaj_placowki'] == 'nie dotyczy')
assert_art_school_names(all_rows.loc[art, 'school_name'])
print(f'{int(art.sum()):,} "nie dotyczy" liceum rows, all with an art-school name')

# Rejected rows: whatever the pipeline drops is written out. Empty text stays ''
# so the CSV reads back equal (keep_default_na=False below).
rejected = (all_rows.loc[~all_rows['outcome'].isin(TYPES),
                         ['year', 'rspo', 'school_name', 'typ_placowki', 'rodzaj_placowki',
                          'outcome']]
            .rename(columns={'outcome': 'reason'})
            .sort_values(['year', 'rspo'], kind='stable')
            .reset_index(drop=True))
for column in ('school_name', 'typ_placowki', 'rodzaj_placowki'):
    rejected[column] = rejected[column].fillna('')

# The population. TERYT is normalised only now: an OKE pseudo-row need not carry one.
df = (all_rows[all_rows['outcome'].isin(TYPES)]
      .rename(columns={'outcome': 'type'})
      .reset_index(drop=True))
df['teryt'] = df['teryt_raw'].map(normalise_teryt)
df = df.drop(columns='teryt_raw')
assert not df[['rspo', 'year']].duplicated().any(), 'duplicate (rspo, year) after the filter'
types_per_school = df.groupby('rspo')['type'].nunique()
assert (types_per_school == 1).all(), (
    f'{int((types_per_school > 1).sum())} schools appear under both types')
voivodeships = df.groupby(['type', 'year'])['wojewodztwo'].nunique()
assert (voivodeships == 16).all(), voivodeships.to_dict()
df = add_level_keys(df)
ALL_YEARS = sorted(int(y) for y in df['year'].unique())

n_rows = df['type'].value_counts()
n_schools = df.drop_duplicates('rspo')['type'].value_counts()
print(f"population: {n_rows['liceum']:,} liceum and {n_rows['technikum']:,} technikum rows; "
      f"{n_schools['liceum']:,} licea and {n_schools['technikum']:,} technika; "
      f'years {ALL_YEARS}')

year                   2023  2024  2025  2026  total
outcome                                             
liceum                 2115  2135  2164  2280   8694
technikum                 0  1700  1699  1703   5102
oke_pseudo_row           15    16    17    17     65
branzowa_ii               0   176   171   200    547
adult_liceum            126   572   640   584   1922
technikum_2023            9     0     0     0      9
missing_basic_subject    57   118   106    91    372
rows in file           2322  4717  4797  4875  16711
335 "nie dotyczy" liceum rows, all with an art-school name
population: 8,694 liceum and 5,102 technikum rows; 2,374 licea and 1,770 technika; years [2023, 2024, 2025, 2026]


In [4]:
# ── Write output/matura/rejected_schools.csv (idempotent) ────────────────────
from export_io import commit_export, dataframes_equal

rejected_path = OUTPUT_DIR / 'rejected_schools.csv'
rejected_unchanged = False
if not FORCE_REGENERATE and rejected_path.exists():
    rejected_unchanged = dataframes_equal(
        rejected, pd.read_csv(rejected_path, keep_default_na=False), verbose=False)
commit_export(
    rejected_path,
    unchanged=rejected_unchanged,
    write=lambda: rejected.to_csv(rejected_path, index=False),
    detail=f'{len(rejected):,} rejected rows',
    force_regenerate=FORCE_REGENERATE,
    write_output_files=WRITE_OUTPUT_FILES,
)
display(rejected.groupby(['reason', 'year']).size().unstack(fill_value=0))

  rejected_schools.csv: no changes, skipping


year,2023,2024,2025,2026
reason,,,,
adult_liceum,126,572,640,584
branzowa_ii,0,176,171,200
missing_basic_subject,57,118,106,91
oke_pseudo_row,15,16,17,17
technikum_2023,9,0,0,0


## 2. Subjects

**Basic level.** Polish, maths and English at the basic level are compulsory, so every
population school has all three (rule 8 of section 1). `composite_min` is their minimum,
exactly as in E8, and never includes an extended subject: no extended subject is present at
every school.

**Extended level.** Students choose their extended subjects, so coverage varies from school
to school. The rule (spec §4.2):

> An extended subject — a `… poziom rozszerzony (M)` column in `egzaminy dodatkowe` — is
> included if at least 100 population licea (youth and art licea) have a published mean in
> it in **every loaded year**.

100 is roughly the size at which the default (voivodeship) reference passes its
≥ 5-schools gate in most voivodeships. CIE publishes a mean only where n ≥ 5, so "has a
published mean" is the count that matters.

**The rule does not select subjects.** The pipeline's subject list has one owner,
`matura.py`'s key map (`EXTENDED_SUBJECTS`). The cell below counts every extended-level
subject in the source files and asserts that the subjects passing the rule are exactly the
mapped ones, in both directions. A breach stops the notebook for a human to decide. German,
at 104 licea in 2025, sits just above the threshold; a weak year must fail loudly here, not
make the subject vanish from the map without anyone noticing.

The count runs over the parsed source frames (`RAW`), not over `df`: `df` carries only the
mapped subjects, and the rule has to see the unmapped ones too.

In [5]:
# ── Coverage of every extended-level subject, counted on population schools ──────
from school_quality.matura import (
    COVERAGE_MIN_LICEA,
    EXTENDED_BAND,
    EXTENDED_SUBJECTS,
    MEASURE_MEAN,
    assert_coverage_matches,
    passing_coverage,
)

EXTENDED_SUFFIX = ' poziom rozszerzony (m)'  # the extended level, exact '(M)' suffix (§2.3)
RSPO_COLUMN = ('meta', '', 'rspo szkoly')
# Print these tables whole: unwrapped, untruncated, full row labels.
WIDE_TABLE = ('display.width', 250, 'display.max_columns', 20, 'display.max_rows', 100,
              'display.max_colwidth', 100)
TYPE_YEARS = {
    school_type: sorted(df.loc[df['type'] == school_type, 'year'].unique().tolist())
    for school_type in ('liceum', 'technikum')
}
# The rule runs over every loaded year, so every loaded file must hold population licea.
assert TYPE_YEARS['liceum'] == ALL_YEARS == sorted(RAW), (TYPE_YEARS, ALL_YEARS, sorted(RAW))


def population_rows(year: int, school_type: str) -> pd.DataFrame:
    """RAW[year]'s rows for one type's population schools in that year.

    RSPO is unique only after the population filter (OKE pseudo-rows reuse one RSPO
    across types), so the match must return exactly one source row per school.
    """
    wanted = set(df.loc[(df['year'] == year) & (df['type'] == school_type), 'rspo'])
    raw = RAW[year]
    rspo = pd.to_numeric(raw[RSPO_COLUMN], errors='raise').astype(int)
    rows = raw[rspo.isin(wanted).to_numpy()]
    assert len(rows) == len(wanted), (
        f'{year} {school_type}: {len(rows)} source rows for {len(wanted)} population schools')
    return rows


def published_means(rows: pd.DataFrame, band: str, subject: str) -> int:
    """Schools in `rows` with a published mean; an empty cell is CIE's n < 5 withholding."""
    means = pd.to_numeric(rows[(band, subject, MEASURE_MEAN)], errors='raise')
    return int(means.notna().sum())


candidates = sorted({
    subject
    for year in ALL_YEARS
    for band, subject, measure in RAW[year].columns
    if band == EXTENDED_BAND and subject.endswith(EXTENDED_SUFFIX) and measure == MEASURE_MEAN
})
coverage = {}  # (type, year) -> {subject: schools with a published mean, None = no column}
for school_type, years in TYPE_YEARS.items():
    for year in years:
        rows = population_rows(year, school_type)
        in_file = {s for b, s, m in RAW[year].columns if b == EXTENDED_BAND and m == MEASURE_MEAN}
        coverage[(school_type, year)] = {
            s: published_means(rows, EXTENDED_BAND, s) if s in in_file else None
            for s in candidates
        }

# The rule counts population licea only; a subject absent from a year's file counts 0 there.
licea_counts = {s: {y: coverage[('liceum', y)][s] or 0 for y in ALL_YEARS} for s in candidates}
passing = passing_coverage(licea_counts, ALL_YEARS)

key_of = {subject: key for key, (_, subject) in EXTENDED_SUBJECTS.items()}
coverage_table = pd.DataFrame(
    {(t, y): [coverage[(t, y)][s] for s in candidates]
     for t, years in TYPE_YEARS.items() for y in years},
    index=pd.Index([s.removesuffix(EXTENDED_SUFFIX) for s in candidates], name='subject'),
).astype('Int64')
coverage_table[('rule', 'licea min')] = [min(licea_counts[s].values()) for s in candidates]
coverage_table[('rule', 'passes')] = [s in passing for s in candidates]
coverage_table[('rule', 'key')] = [key_of.get(s, '') for s in candidates]
coverage_table = coverage_table.sort_values(('rule', 'licea min'), ascending=False,
                                            kind='stable')

print(f'{len(candidates)} "...{EXTENDED_SUFFIX}" subjects in "{EXTENDED_BAND}", '
      f'{ALL_YEARS[0]}-{ALL_YEARS[-1]}: population schools with a published mean, per type '
      f'and year (<NA> = no such column in the file for that year)')
with pd.option_context(*WIDE_TABLE):
    display(coverage_table)

print(f'Pass the rule (>= {COVERAGE_MIN_LICEA} population licea in every loaded year): '
      f'{len(passing)}')
assert_coverage_matches(passing)  # raises ValueError naming both differences
print(f'matura.py key map and coverage rule agree: the {len(EXTENDED_SUBJECTS)} mapped '
      f'subjects are exactly the ones that pass.')

28 "... poziom rozszerzony (m)" subjects in "egzaminy dodatkowe", 2023-2026: population schools with a published mean, per type and year (<NA> = no such column in the file for that year)


liceum                   technikum                  rule                          
                                           2023  2024  2025  2026      2024  2025  2026 licea min passes                key
subject                                                                                                                    
jezyk angielski                            2012  2036  2090  2221      1548  1552  1589      2012   True        angielski_r
jezyk polski                               1557  1537  1537  1704       600   470   513      1537   True           polski_r
geografia                                  1492  1486  1530  1773      1043   976  1067      1486   True        geografia_r
biologia                                   1479  1421  1396  1759       446   394   568      1396   True         biologia_r
matematyka                                 1369  1263  1289  1505       913   801   900      1263   True       matematyka_r
chemia                                     1008   908   893  1154       116    79   122       893   True           chemia_r
historia                                    935   837   810  1109       127    96   120       810   True         historia_r
fizyka                                      651   589   596   702       231   157   200       589   True           fizyka_r
informatyka                                 337   284   274   325       266   218   252       274   True      informatyka_r
historia sztuki                             112   147   133   207         8     6    15       112   True  historia_sztuki_r
jezyk niemiecki                             161   114   104   127        18    10     7       104   True        niemiecki_r
jezyk rosyjski                               33    36    33    70         5    21    32        33  False                   
jezyk hiszpanski                             32    39    50    70         0     0     0        32  False                   
filozofia                                    31    34    45    39         8     5     2        31  False                   
jezyk francuski                              15    16    13    21         0     0     0        13  False                   
historia muzyki                               7     8     9     9         0     0     0         7  False                   
jezyk lacinski i kultura antyczna             5     6     4     6         0     0     0         4  False                   
jezyk wloski                                  4     7     6     9         0     0     0         4  False                   
jezyk mniejszosci narodowej - bialoruski      2     2     1     1         0     0     0         1  False                   
jezyk mniejszosci narodowej - litewski        1     1     1     1         0     0     0         1  False                   
jezyk kaszubski                               1     0     2     0         0     0     0         0  False                   
jezyk lemkowski                               0     0     0     0         0     0     0         0  False                   
jezyk mniejszosci narodowej - czeski       <NA>     0     0     0         0     0     0         0  False                   
jezyk mniejszosci narodowej - hebrajski    <NA>  <NA>     0     0      <NA>     0     0         0  False                   
jezyk mniejszosci narodowej - niemiecki    <NA>     0     0  <NA>         0     0  <NA>         0  False                   
jezyk mniejszosci narodowej - ukrainski       0     1     0     0         0     0     0         0  False                   
jezyk ukrainski                            <NA>  <NA>  <NA>     4      <NA>  <NA>     1         0  False                   
wiedza o spoleczenstwie                     826   759   789  <NA>       218   197  <NA>         0  False

Pass the rule (>= 100 population licea in every loaded year): 11
matura.py key map and coverage rule agree: the 11 mapped subjects are exactly the ones that pass.


### What the rule does not see

Three exclusions follow from the rule's wording rather than from size (spec §4.2):

- **Civics** (`wiedza o spoleczenstwie`) would pass on size until 2025, but in 2026 it
  splits into two exams by curriculum, `(M19-21)` and `(M22)`. Neither ends in exactly
  `(M)`, so the rule finds no civics column in 2026 (`<NA>` in the table above) and civics
  fails. Joining two exams into one subject is its own design problem, out of scope.
- **The bilingual level** (`poziom dwujęzyczny`) is a different exam level, not an extended
  one, so it is excluded by level whatever its size.
- **The `zestawy zadan w jezyku obcym` band** (an extended subject sat in a foreign
  language) is not mapped, so extended results leave those students out. The cell below
  measures how many that is for extended maths.

In [6]:
# ── Civics, the bilingual level and the foreign-language band, per year ─────────
from school_quality.matura import MEASURE_N

CIVICS = 'wiedza o spoleczenstwie poziom rozszerzony'  # one exam to 2025, two in 2026
BILINGUAL_ENGLISH = 'jezyk angielski poziom dwujezyczny (m)'
FOREIGN_LANGUAGE_BAND = 'zestawy zadan w jezyku obcym'
EXTENDED_MATHS = EXTENDED_SUBJECTS['matematyka_r'][1]
FOREIGN_MATHS = 'maths in a foreign language'

left_out = {}  # row label -> {year: formatted value}
for year in ALL_YEARS:
    columns = RAW[year].columns
    rows_by_type = {t: population_rows(year, t) for t, years in TYPE_YEARS.items()
                    if year in years}
    for subject in sorted({s for b, s, m in columns if b == EXTENDED_BAND and m == MEASURE_MEAN
                           and (s.startswith(CIVICS) or s == BILINGUAL_ENGLISH)}):
        label = ('English bilingual' if subject == BILINGUAL_ENGLISH
                 else f'civics {subject.removeprefix(CIVICS).strip()}')
        for t, rows in rows_by_type.items():
            count = published_means(rows, EXTENDED_BAND, subject)
            left_out.setdefault(f'{label}, {t}', {})[year] = f'{count:,}'

    # Takers, not means: the foreign-language sets are small and mostly unpublished.
    both = pd.concat(rows_by_type.values())
    foreign_columns = [(b, s, m) for b, s, m in columns
                       if b == FOREIGN_LANGUAGE_BAND and s.startswith('matematyka ')
                       and m == MEASURE_N]
    assert foreign_columns, f'{year}: no extended maths in "{FOREIGN_LANGUAGE_BAND}"'
    foreign = sum(pd.to_numeric(both[c], errors='raise').fillna(0) for c in foreign_columns)
    main = pd.to_numeric(both[(EXTENDED_BAND, EXTENDED_MATHS, MEASURE_N)],
                         errors='raise').fillna(0)
    share = 100 * foreign.sum() / (foreign.sum() + main.sum())
    left_out.setdefault(f'{FOREIGN_MATHS}: schools', {})[year] = f'{int((foreign > 0).sum()):,}'
    left_out.setdefault(f'{FOREIGN_MATHS}: takers', {})[year] = f'{int(foreign.sum()):,}'
    left_out.setdefault(f'{FOREIGN_MATHS}: % of extended-maths takers', {})[year] = (
        f'{share:.1f}')

left_out_table = pd.DataFrame(left_out).T.sort_index().reindex(columns=ALL_YEARS).fillna('—')
print('Population schools per year; "—" = no such column that year, or no population '
      'schools of that type')
with pd.option_context(*WIDE_TABLE):
    display(left_out_table)

Population schools per year; "—" = no such column that year, or no population schools of that type


,2023,2024,2025,2026
"English bilingual, liceum",172,203,289,402
"English bilingual, technikum",—,25,23,48
"civics (m), liceum",826,759,789,—
"civics (m), technikum",—,218,197,—
"civics (m19-21), liceum",—,—,—,2
"civics (m19-21), technikum",—,—,—,253
"civics (m22), liceum",—,—,—,998
"civics (m22), technikum",—,—,—,1
maths in a foreign language: % of extended-maths takers,1.4,1.2,1.2,1.4
maths in a foreign language: schools,46,59,68,88


## 3. Why compare within type

Scored against one shared population, the colour mostly encodes school type. The cell below
scores one year — 2025, the year spec §5.1 quotes — two ways:

- **shared reference**: every school against the mean of all population schools of that
  year, licea and technika together;
- **within type**: each school against its own type only, with the class boundaries
  computed per type too.

The score is the map's default composite, `composite_min` of the three basic subjects'
`unit_norm_diff_mean`, at the **national** reference, sorted into the map's three classes
at ±0.33σ around the composite's own mean (section 5 re-measures whether that band suits
matura).

In [7]:
# ── Shared reference vs within type: colour classes, one year (spec §5.1) ───────
import numpy as np

from school_quality.levels import attach_reference
from school_quality.matura import BASIC_KEYS

WITHIN_TYPE_YEAR = 2025  # the year spec §5.1 quotes; both types sit EM2023 from 2024
assert all(WITHIN_TYPE_YEAR in years for years in TYPE_YEARS.values()), TYPE_YEARS
year_rows = df[df['year'] == WITHIN_TYPE_YEAR]
is_liceum = year_rows['type'] == 'liceum'


def composite_vs_national(rows: pd.DataFrame) -> pd.Series:
    """composite_min of the basic subjects' unit_norm_diff_mean against the national
    reference of `rows` (the mean of their school means, per year)."""
    per_subject = {}
    for subject in BASIC_KEYS:
        with_ref = attach_reference(rows, f'mean_{subject}', 'national', 'reference', 'mean')
        diff = with_ref[f'mean_{subject}'] - with_ref['reference']
        ref = with_ref['reference']
        per_subject[subject] = np.where(diff >= 0, diff / (100 - ref), diff / ref)
    return pd.DataFrame(per_subject, index=rows.index).min(axis=1)


def colour_classes(scores: pd.Series) -> pd.Series:
    """The map's three classes around the scores' own mean (composite_min's centre)."""
    centre, sigma = scores.mean(), scores.std()
    labels = np.select([scores > centre + 0.33 * sigma, scores < centre - 0.33 * sigma],
                       ['above (A)', 'below (C)'], 'around (B)')
    return pd.Series(labels, index=scores.index)


shared = colour_classes(composite_vs_national(year_rows))
within = pd.concat([colour_classes(composite_vs_national(year_rows[is_liceum])),
                    colour_classes(composite_vs_national(year_rows[~is_liceum]))])
class_shares = {}
for reference, classes in (('shared reference', shared), ('within type', within)):
    for school_type, plural in (('liceum', 'licea'), ('technikum', 'technika')):
        of_type = classes.loc[year_rows.index[year_rows['type'] == school_type]]
        class_shares[f'{plural}, {reference}'] = of_type.value_counts(normalize=True) * 100
class_table = (pd.DataFrame(class_shares).T
               .reindex(columns=['above (A)', 'around (B)', 'below (C)'])
               .map(lambda v: f'{v:.1f}%'))
print(f'{WITHIN_TYPE_YEAR}: share of each type in each colour class '
      f'({int(is_liceum.sum()):,} licea, {int((~is_liceum).sum()):,} technika)')
display(class_table)

2025: share of each type in each colour class (2,164 licea, 1,699 technika)


,above (A),around (B),below (C)
"licea, shared reference",53.3%,21.6%,25.0%
"technika, shared reference",8.2%,30.8%,61.0%
"licea, within type",36.4%,26.0%,37.6%
"technika, within type",37.0%,27.5%,35.5%


### How far apart the two types sit

Per basic subject, on the raw school means of the same year: the gap between the two
types' mean school results, in units of the standard deviation of all secondary schools
taken together and, for comparison, of the pooled within-type standard deviation; the share
of technika above the median liceum; and the share of the between-school variance that
school type alone explains.

In [8]:
# ── Separation of the two types per basic subject (raw school means) ────────────
separation = {}
for subject in BASIC_KEYS:
    means = year_rows[f'mean_{subject}']
    lic, tech = means[is_liceum], means[~is_liceum]
    gap = lic.mean() - tech.mean()
    pooled_within_sd = np.sqrt(((len(lic) - 1) * lic.var() + (len(tech) - 1) * tech.var())
                               / (len(means) - 2))
    type_mean = means.groupby(year_rows['type']).transform('mean')
    explained = ((type_mean - means.mean()) ** 2).sum() / ((means - means.mean()) ** 2).sum()
    separation[subject] = {
        'mean of school means, licea': f'{lic.mean():.1f}',
        'mean of school means, technika': f'{tech.mean():.1f}',
        'gap / SD of all schools': f'{gap / means.std():.2f}',
        'gap / pooled within-type SD': f'{gap / pooled_within_sd:.2f}',
        'technika above the median liceum': f'{(tech > lic.median()).mean():.1%}',
        'variance explained by type alone': f'{explained:.1%}',
    }
print(f'{WITHIN_TYPE_YEAR}, population schools')
display(pd.DataFrame(separation))

2025, population schools


,polski,matematyka,angielski
"mean of school means, licea",61.3,61.6,80.1
"mean of school means, technika",50.8,47.2,66.5
gap / SD of all schools,0.94,0.84,0.96
gap / pooled within-type SD,1.06,0.93,1.09
technika above the median liceum,7.9%,12.8%,6.8%
variance explained by type alone,21.7%,17.5%,22.7%


**Takeaway.** Against a shared reference, most licea land above average and most technika
below — a map coloured that way would mostly show which schools are licea. Within type,
both types spread across the three classes in similar proportions, so the colour says
something about a school compared with schools like it. That is why each type is compared
with its own type by default, and the combined `all` population (from 2024) stays available
for the cross-type question, where raw `mean` and pass rate remain directly comparable.

The data cannot say *why* the types differ, and the map must not suggest a cause
(`MAP_APP_BRIEF.md` §7; `repo/CLAUDE.md`, "outcome, not value-added"). Structural facts a
reader should know, not causes we can attribute: licea and technika teach different
curricula (technika also teach a trade); technika students also sit vocational
qualification exams, absent from this data; and the matura is voluntary.

## 4. Choosing the metric

E8 chose its metric by leave-one-out (LOO) stability (E8 notebook, section 3): for a
school with N years, compute the score N times, each time leaving one year out; the
closer those N estimates sit, the less the score depends on which year happened to
be included. The spread is the LOO standard deviation, divided by the metric's
cross-school spread so metrics on different scales compare. 8 per-year metrics × 5
across-year aggregations are tested per subject.

This re-runs the test on matura data **per type**, with references within type at
E8's primary level (same year, same voivodeship). The folds are thin: licea have 4
years, so 4 folds of 3 years; technika have 3, so 3 folds of 2. Read the tables as
evidence about the same choice E8 made, not as a fresh selection. The export carries
all four candidate metrics regardless, and `PRIMARY_METRIC` stays hand-set in
section 0; if the matura winner differs from E8's (`unit_norm_diff_mean`, weighted
by cohort), that is a finding for the user (spec §5.4).

In [9]:
# ── Leave-one-out stability per type: 8 per-year metrics × 5 aggregations ─────
import numpy as np
from scipy import stats

from school_quality.matura import BASIC_KEYS

CANDIDATE_METRICS = ['mean', 'median', 'diff_mean', 'diff_median', 'unit_norm_diff_mean',
                     'unit_norm_diff_median', 'pct_mean', 'pct_median']
AGG_METHODS = ['mean', 'mean_weighted_by_students_count', 'mean_weighted_by_students_count_sqrt',
               'median', 'trimmed_mean']
REGION = ['year', 'teryt_wojewodztwo']   # same year, same voivodeship, within one type


def unit_norm(diff, reference):
    return np.where(diff >= 0, diff / (100 - reference), diff / reference)


def per_year_metrics(rows, subject):
    """The 8 candidate per-year values of one basic subject, for one type's rows.

    The reference is the mean of school means (median of school medians) of the same
    type in the same year and voivodeship; pct_* is the percentile rank there.
    """
    groups = [rows[column] for column in REGION]
    out = pd.DataFrame({'rspo': rows['rspo'], 'year': rows['year'], 'n': rows[f'n_{subject}']})
    for stat in ('mean', 'median'):
        value = rows[f'{stat}_{subject}']
        reference = value.groupby(groups).transform(stat)
        out[stat] = value
        out[f'diff_{stat}'] = value - reference
        out[f'unit_norm_diff_{stat}'] = unit_norm(value - reference, reference)
        out[f'pct_{stat}'] = value.groupby(groups).transform(
            lambda x: stats.rankdata(x.fillna(x.median()), method='average') / len(x) * 100)
    return out


def fold_scores(values, weights, method):
    """LOO fold scores for (schools, years) arrays with no gaps: column i leaves year i out.

    trimmed_mean drops the best and worst year; with two years left it is the mean,
    as in E8.
    """
    folds = np.empty(values.shape, dtype=float)
    for i in range(values.shape[1]):
        v = np.delete(values, i, axis=1)
        w = np.delete(weights, i, axis=1)
        if method == 'mean':
            folds[:, i] = v.mean(axis=1)
        elif method == 'median':
            folds[:, i] = np.median(v, axis=1)
        elif method == 'trimmed_mean':
            folds[:, i] = (v.mean(axis=1) if v.shape[1] <= 2
                           else np.sort(v, axis=1)[:, 1:-1].mean(axis=1))
        else:
            ww = w if method == 'mean_weighted_by_students_count' else np.sqrt(w)
            folds[:, i] = (v * ww).sum(axis=1) / ww.sum(axis=1)
    return folds


def complete_rows(school_type):
    """The type's rows for schools with every one of the type's years (E8: rspo_all_years)."""
    rows = df[df['type'] == school_type]
    years = sorted(int(y) for y in rows['year'].unique())
    complete = rows.groupby('rspo')['year'].nunique() == len(years)
    return rows[rows['rspo'].isin(complete[complete].index)], years, int(complete.sum())


stability = []
for school_type in ('liceum', 'technikum'):
    rows, years, n_schools = complete_rows(school_type)
    print(f'{school_type}: {len(years)} years {years}; {n_schools:,} schools with every year, '
          f'so {len(years)} folds of {len(years) - 1} years each')
    for subject in BASIC_KEYS:
        metrics = per_year_metrics(rows, subject)
        weights = metrics.pivot(index='rspo', columns='year', values='n').to_numpy(dtype=float)
        for metric in CANDIDATE_METRICS:
            values = metrics.pivot(index='rspo', columns='year', values=metric)
            values = values.to_numpy(dtype=float)
            assert not np.isnan(values).any(), (school_type, subject, metric)
            spread = metrics[metric].std()   # cross-school spread of the per-year value
            for method in AGG_METHODS:
                loo_std = np.std(fold_scores(values, weights, method), axis=1)
                stability.append({'type': school_type, 'subject': subject, 'metric': metric,
                                  'aggregation': method,
                                  'relative LOO std': float(np.median(loo_std) / spread)})
stability = pd.DataFrame(stability)

for (school_type, subject), table in stability.groupby(['type', 'subject'], sort=False):
    wide = (table.pivot(index='metric', columns='aggregation', values='relative LOO std')
            .reindex(index=CANDIDATE_METRICS, columns=AGG_METHODS))
    best_metric, best_method = wide.stack().idxmin()
    e8_choice = wide.loc['unit_norm_diff_mean', 'mean_weighted_by_students_count']
    print(f'\n{school_type}, {subject}: lowest median relative LOO std = '
          f'{wide.stack().min():.4f} for {best_metric} / {best_method}; '
          f"E8's choice (unit_norm_diff_mean / weighted) = {e8_choice:.4f}")
    display(wide.round(4))

liceum: 4 years [2023, 2024, 2025, 2026]; 1,966 schools with every year, so 4 folds of 3 years each
technikum: 3 years [2024, 2025, 2026]; 1,624 schools with every year, so 3 folds of 2 years each



liceum, polski: lowest median relative LOO std = 0.1066 for pct_mean / median; E8's choice (unit_norm_diff_mean / weighted) = 0.1216


aggregation,mean,mean_weighted_by_students_count,mean_weighted_by_students_count_sqrt,median,trimmed_mean
metric,,,,,
mean,0.1168,0.1184,0.1182,0.1111,0.1111
median,0.1272,0.1254,0.1267,0.1361,0.1361
diff_mean,0.1154,0.1162,0.1159,0.1137,0.1137
diff_median,0.1279,0.1258,0.1266,0.1168,0.1168
unit_norm_diff_mean,0.1221,0.1216,0.1217,0.1162,0.1162
unit_norm_diff_median,0.1327,0.1297,0.1315,0.1214,0.1214
pct_mean,0.1215,0.1215,0.1211,0.1066,0.1066
pct_median,0.1322,0.1289,0.1315,0.1145,0.1145



liceum, matematyka: lowest median relative LOO std = 0.0533 for pct_mean / median; E8's choice (unit_norm_diff_mean / weighted) = 0.0626


aggregation,mean,mean_weighted_by_students_count,mean_weighted_by_students_count_sqrt,median,trimmed_mean
metric,,,,,
mean,0.0716,0.0737,0.0726,0.0666,0.0666
median,0.0773,0.0794,0.0779,0.0774,0.0774
diff_mean,0.0650,0.0628,0.0638,0.0580,0.0580
diff_median,0.0737,0.0728,0.0737,0.0655,0.0655
unit_norm_diff_mean,0.0632,0.0626,0.0627,0.0577,0.0577
unit_norm_diff_median,0.0687,0.0682,0.0682,0.0618,0.0618
pct_mean,0.0624,0.0615,0.0622,0.0533,0.0533
pct_median,0.0679,0.0681,0.0687,0.0557,0.0557


...

aggregation,mean,mean_weighted_by_students_count,mean_weighted_by_students_count_sqrt,median,trimmed_mean
metric,,,,,
mean,0.2300,0.2280,0.2311,0.2300,0.2300
median,0.2356,0.2383,0.2378,0.2356,0.2356
diff_mean,0.2202,0.2201,0.2210,0.2202,0.2202
diff_median,0.2309,0.2232,0.2276,0.2309,0.2309
unit_norm_diff_mean,0.2211,0.2196,0.2216,0.2211,0.2211
unit_norm_diff_median,0.2312,0.2258,0.2298,0.2312,0.2312
pct_mean,0.2317,0.2287,0.2318,0.2317,0.2317
pct_median,0.2426,0.2439,0.2436,0.2426,0.2426



technikum, matematyka: lowest median relative LOO std = 0.1251 for diff_mean / mean_weighted_by_students_count; E8's choice (unit_norm_diff_mean / weighted) = 0.1281


aggregation,mean,mean_weighted_by_students_count,mean_weighted_by_students_count_sqrt,median,trimmed_mean
metric,,,,,
mean,0.1621,0.1586,0.1609,0.1621,0.1621
median,0.1655,0.1629,0.1637,0.1655,0.1655
diff_mean,0.1280,0.1251,0.1254,0.1280,0.1280
diff_median,0.1380,0.1336,0.1363,0.1380,0.1380
unit_norm_diff_mean,0.1301,0.1281,0.1295,0.1301,0.1301
unit_norm_diff_median,0.1394,0.1365,0.1394,0.1394,0.1394
pct_mean,0.1319,0.1310,0.1312,0.1319,0.1319
pct_median,0.1393,0.1378,0.1386,0.1393,0.1393



technikum, angielski: lowest median relative LOO std = 0.1381 for unit_norm_diff_mean / mean_weighted_by_students_count; E8's choice (unit_norm_diff_mean / weighted) = 0.1381


aggregation,mean,mean_weighted_by_students_count,mean_weighted_by_students_count_sqrt,median,trimmed_mean
metric,,,,,
mean,0.1540,0.1500,0.1531,0.1540,0.1540
median,0.1540,0.1563,0.1543,0.1540,0.1540
diff_mean,0.1486,0.1469,0.1465,0.1486,0.1486
diff_median,0.1568,0.1585,0.1573,0.1568,0.1568
unit_norm_diff_mean,0.1447,0.1381,0.1420,0.1447,0.1447
unit_norm_diff_median,0.1586,0.1560,0.1593,0.1586,0.1586
pct_mean,0.1531,0.1479,0.1509,0.1531,0.1531
pct_median,0.1632,0.1624,0.1621,0.1632,0.1632


## 5. Final metric and colours

`PRIMARY_METRIC` is `unit_norm_diff_mean` aggregated by cohort-weighted mean, as in E8,
unless the section above gives the user reason to change it. The map's three colour
classes sit at ±0.33σ around the centre; σ, centre, p1 and p99 are computed in
section 6 per (population, level, metric, subject).

E8 justifies the ±0.33σ band by measuring the base score's own year-to-year noise
at about 0.12σ: the band is wider than the noise, so the three classes are
distinguishable. That is an E8 measurement. The cell below measures it for matura,
per type and basic subject: the median over schools of the LOO standard deviation
of the weighted `unit_norm_diff_mean` base score, divided by σ of the base score
across those schools. A ratio reaching 0.33 is a finding to bring back (spec §12),
not to absorb.

In [10]:
# ── Year-to-year noise of the base score vs the ±0.33σ class band (spec §5.6) ─
noise = []
for school_type in ('liceum', 'technikum'):
    rows, _, _ = complete_rows(school_type)
    for subject in BASIC_KEYS:
        metrics = per_year_metrics(rows, subject)
        values = metrics.pivot(index='rspo', columns='year', values='unit_norm_diff_mean')
        values = values.to_numpy(dtype=float)
        weights = metrics.pivot(index='rspo', columns='year', values='n').to_numpy(dtype=float)
        base = (values * weights).sum(axis=1) / weights.sum(axis=1)
        loo = fold_scores(values, weights, 'mean_weighted_by_students_count')
        school_noise = loo.std(axis=1, ddof=1)   # as E8's section 3 (a pandas std)
        sigma = base.std(ddof=1)
        noise.append({'type': school_type, 'subject': subject, 'schools': len(base),
                      'median LOO std': round(float(np.median(school_noise)), 4),
                      'σ of base score': round(float(sigma), 4),
                      'noise / σ': round(float(np.median(school_noise) / sigma), 3)})
noise = pd.DataFrame(noise)
display(noise)
worst = float(noise['noise / σ'].max())
print(f'largest noise / σ: {worst:.3f} (E8: about 0.12); the ±0.33σ band is '
      f'{0.33 / worst:.1f}× the noisiest type and subject')
if worst >= 0.33:
    print('WARNING: year-to-year noise reaches the ±0.33σ band. This is a finding to bring '
          'back to the user (spec §12), not to absorb.')

,type,subject,schools,median LOO std,σ of base score,noise / σ
0,liceum,polski,1966,0.0298,0.1900,0.157
1,liceum,matematyka,1966,0.0253,0.3377,0.075
2,liceum,angielski,1966,0.0260,0.3429,0.076
3,technikum,polski,1624,0.0449,0.1365,0.329
4,technikum,matematyka,1624,0.0385,0.2271,0.169
5,technikum,angielski,1624,0.0404,0.2190,0.184


largest noise / σ: 0.329 (E8: about 0.12); the ±0.33σ band is 1.0× the noisiest type and subject


## 6. Export

Three self-contained datasets, one per comparison population, each in E8's exact shape
(spec §6.1): `docs/data/matura/{liceum,technikum,all}/`. The analyst files go to
`output/matura/`. Subsection 6a computes every score; 6b–6d only write files.

### 6a. Identity, references, views, ranks

**Identity is built once** over every population row: everything from the school's
latest year except the address, which `select_address` picks over all its years (the
E8 rule, which keeps a fuller older address when a newer year only shortens it). A
school therefore has one address in every dataset.

**Per population**, references are the mean of school means of that population in the
same year — nationally, or in the voivodeship. A reference's size is the number of
schools **with a published mean in that subject** (spec §5.5 a): counting every school
would publish scores for extended subjects from references of fewer than five scored
schools. A school's multi-year score weights each year by that subject's own takers
(§5.5 b). A difference-metric view is withheld when any year it uses has a reference
below 5; a percentile is withheld when the view ranks fewer than 8 schools (§5.5 d).

In [11]:
# ── School identity, once for all populations ────────────────────────────────
from export_io import commit_export, dataframes_equal
from school_quality.address import dropped_words, select_address

selected_addresses, declined_addresses = {}, []
for rspo, group in df.groupby('rspo'):
    selected_addresses[rspo], group_declined = select_address(group)
    declined_addresses.extend(group_declined)

IDENTITY = (df.sort_values(['year', 'rspo'], ascending=[False, True])
            .drop_duplicates('rspo', keep='first')
            .set_index('rspo')
            [['type', 'school_name', 'is_public', 'miejscowosc', 'ulica_nr', 'powiat', 'gmina',
              'typ_gminy', 'wojewodztwo', 'teryt', 'teryt_wojewodztwo', 'teryt_powiat',
              'teryt_gmina']]
            .copy())
IDENTITY['miejscowosc'] = [selected_addresses[r][0] for r in IDENTITY.index]
IDENTITY['ulica_nr'] = [selected_addresses[r][1] for r in IDENTITY.index]
IDENTITY = IDENTITY.reset_index().sort_values('rspo').reset_index(drop=True)
assert IDENTITY['rspo'].is_unique
n_types = IDENTITY['type'].value_counts()
print(f"identity: {len(IDENTITY):,} schools ({n_types['liceum']:,} licea, "
      f"{n_types['technikum']:,} technika)")

declined = pd.DataFrame(declined_addresses, columns=[
    'rspo', 'school_name', 'year', 'kept_miejscowosc', 'kept_ulica_nr',
    'declined_miejscowosc', 'declined_ulica_nr'])
declined['dropped_words'] = [dropped_words(kept, gone) for kept, gone
                             in zip(declined['kept_ulica_nr'], declined['declined_ulica_nr'],
                                    strict=True)]
declined = declined.sort_values(['rspo', 'year']).reset_index(drop=True)
declined_path = OUTPUT_DIR / 'rejected_addresses.csv'
declined_unchanged = False
if not FORCE_REGENERATE and declined_path.exists():
    # keep_default_na=False: an empty dropped_words (13 rows) must read back as '',
    # not NaN, or the comparison fails and the file is rewritten on every run.
    declined_unchanged = dataframes_equal(
        declined, pd.read_csv(declined_path, keep_default_na=False), verbose=False)
commit_export(
    declined_path,
    unchanged=declined_unchanged,
    write=lambda: declined.to_csv(declined_path, index=False),
    detail=f'{len(declined):,} declined address updates',
    force_regenerate=FORCE_REGENERATE,
    write_output_files=WRITE_OUTPUT_FILES,
)

identity: 4,144 schools (2,374 licea, 1,770 technika)
  rejected_addresses.csv: no changes, skipping


'skipped'

In [12]:
# ── Per population: references (rule a: size = published means) ─────────────
import numpy as np

from school_quality.levels import REFERENCE_LEVELS, attach_reference
from school_quality.matura import POPULATIONS, SUBJECT_KEYS, in_population


def SOURCE_LEVEL(metric, level):
    """The level a metric is stored at: mean/median have no reference population."""
    return PRIMARY_REFERENCE_LEVEL if metric in ('mean', 'median') else level


POP_FRAMES = {}
for population in POPULATIONS:
    keep = [in_population(population, t, y) for t, y in zip(df['type'], df['year'], strict=True)]
    frame = df[keep].copy()
    for level in MATURA_LEVELS:
        group_keys = ['year'] + REFERENCE_LEVELS[level]
        for s in SUBJECT_KEYS:
            frame = attach_reference(frame, f'mean_{s}', level, f'ref_mean_{level}_{s}', 'mean')
            frame[f'ref_n_{level}_{s}'] = (frame.groupby(group_keys)[f'mean_{s}']
                                           .transform('count'))
            diff = frame[f'mean_{s}'] - frame[f'ref_mean_{level}_{s}']
            ref = frame[f'ref_mean_{level}_{s}']
            frame[f'diff_mean_{level}_{s}'] = diff
            frame[f'unit_norm_diff_mean_{level}_{s}'] = np.where(
                diff >= 0, diff / (100 - ref).where(100 - ref > 0, 1), diff / ref.where(ref > 0, 1))
            # A broadcast that went wrong yields NaN references silently; a school with
            # a published mean always has one.
            has_mean = frame[f'mean_{s}'].notna()
            assert frame.loc[has_mean, f'ref_mean_{level}_{s}'].notna().all(), (population, level, s)
    POP_FRAMES[population] = frame.copy()

/var/folders/29/29c0wxvn19n1y9df2chjj5700000gq/T/ipykernel_51229/2297693402.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[f'diff_mean_{level}_{s}'] = diff
/var/folders/29/29c0wxvn19n1y9df2chjj5700000gq/T/ipykernel_51229/2297693402.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[f'unit_norm_diff_mean_{level}_{s}'] = np.where(
/Users/mikolaj.olszewski/Projects/Personal/8th grade exams/repo/src/school_quality/levels.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of cal

/var/folders/29/29c0wxvn19n1y9df2chjj5700000gq/T/ipykernel_51229/2297693402.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[f'diff_mean_{level}_{s}'] = diff
/var/folders/29/29c0wxvn19n1y9df2chjj5700000gq/T/ipykernel_51229/2297693402.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[f'unit_norm_diff_mean_{level}_{s}'] = np.where(
/Users/mikolaj.olszewski/Projects/Personal/8th grade exams/repo/src/school_quality/levels.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of cal

In [13]:
# ── Per population: views, ranks, the publication rule, n_ranked ─────────────
from school_quality.aggregate import REFERENCED_METRICS
from school_quality.matura import (
    BASIC_KEYS,
    assert_population_exclusions,
    export_subjects,
)
from school_quality.suppression import suppress_diff_score
from school_quality.views import (
    aggregate_years,
    composite_views,
    rank_view,
    school_views,
    view_param_key,
)


def school_years_for_subject(frame, metric, subject, level):
    """{rspo: {year: (takers, value, reference size)}} over the years with a value."""
    n_col = f'n_{subject}'
    value_col = (f'{metric}_{subject}' if metric in ('mean', 'median')
                 else f'{metric}_{level}_{subject}')
    rows = frame[frame[n_col].notna() & (frame[n_col] > 0) & frame[value_col].notna()]
    rows = rows.sort_values(['rspo', 'year'])
    out = {}
    for rspo, year, n, value, ref_n in zip(rows['rspo'], rows['year'], rows[n_col],
                                           rows[value_col], rows[f'ref_n_{level}_{subject}'],
                                           strict=True):
        out.setdefault(int(rspo), {})[int(year)] = (n, value, int(ref_n))
    return out


VIEWS, POP_SUBJECTS, N_RANKED = {}, {}, {}
withheld_report, small_views, empty_views = {}, set(), set()
for population, frame in POP_FRAMES.items():
    view_results, view_n, candidates = {}, {}, set()
    for level in MATURA_LEVELS:
        for metric in EXPORT_METRICS:
            if SOURCE_LEVEL(metric, level) != level:
                continue   # mean and median exist once, at the primary level
            for subject in SUBJECT_KEYS:
                for rspo, year_map in school_years_for_subject(frame, metric, subject,
                                                               level).items():
                    for view_kind, view_param, years in school_views(sorted(year_map)):
                        candidates.add((level, metric, subject, view_kind, view_param))
                        score, n_students, reference_n = aggregate_years(metric, year_map, years)
                        if metric in REFERENCED_METRICS:
                            score = suppress_diff_score(score, reference_n)
                        if score is None or np.isnan(score):
                            continue
                        key = (level, metric, subject, view_kind, view_param)
                        view_results.setdefault(key, {})[rspo] = score
                        view_n.setdefault(key, {})[rspo] = n_students
            # composite_min over the three basic subjects, in the order the basic
            # subjects' views were stored (views.composite_views says why).
            composite_results, composite_cohorts = composite_views(
                view_results, view_n, level, metric, BASIC_KEYS)
            view_results.update(composite_results)
            view_n.update(composite_cohorts)

    records = []
    for (level, metric, subject, view_kind, view_param), score_map in view_results.items():
        cohort = view_n[(level, metric, subject, view_kind, view_param)]
        for rspo, score, rank, pct, n_in_view in rank_view(score_map):
            records.append({
                'rspo': rspo, 'reference_level': level, 'metric': metric, 'subject': subject,
                'view_kind': view_kind, 'view_param': view_param, 'score': score,
                'rank_overall': rank, 'pct_overall': np.nan if pct is None else pct,
                'n_in_view': n_in_view, 'n_students': int(cohort[rspo]),
            })
    views = pd.DataFrame(records)
    assert not views[['rspo', 'reference_level', 'metric', 'subject', 'view_kind',
                      'view_param']].duplicated().any(), population

    # The per-population publication rule (spec §4.2): a subject with a scale entry
    # below 2 base scores must be exactly the population's POPULATION_EXCLUSIONS.
    base_counts = (views[views['view_kind'] == 'base']
                   .groupby(['reference_level', 'metric', 'subject']).size())
    short = {s for s in SUBJECT_KEYS for level in MATURA_LEVELS for metric in EXPORT_METRICS
             if base_counts.get((SOURCE_LEVEL(metric, level), metric, s), 0) < 2}
    assert_population_exclusions(population, short)
    POP_SUBJECTS[population] = export_subjects(population)
    views = views[views['subject'].isin(POP_SUBJECTS[population])].reset_index(drop=True)
    VIEWS[population] = views

    # n_ranked[level][metric][subject][view_kind]: level-keyed like the rest of
    # scale.json, mean/median repeating their voivodeship values under national.
    keys = views.drop_duplicates(['reference_level', 'metric', 'subject', 'view_kind',
                                  'view_param'])
    n_ranked = {}
    for level in MATURA_LEVELS:
        for metric in EXPORT_METRICS:
            at = keys[(keys['reference_level'] == SOURCE_LEVEL(metric, level))
                      & (keys['metric'] == metric)]
            for row in at.itertuples(index=False):
                entry = (n_ranked.setdefault(level, {}).setdefault(metric, {})
                         .setdefault(row.subject, {}))
                if row.view_kind == 'base':
                    entry['base'] = int(row.n_in_view)
                else:
                    entry.setdefault(row.view_kind, {})[
                        view_param_key(row.view_kind, row.view_param)] = int(row.n_in_view)
    N_RANKED[population] = n_ranked

    # What spec §5.5 a and d say to expect, recounted on the published views.
    published = POP_SUBJECTS[population]
    ranked_keys = set(view_results)
    difference = {'diff_mean', 'unit_norm_diff_mean'}
    for (level, metric, subject, view_kind, view_param) in candidates - ranked_keys:
        if subject in published and metric == 'unit_norm_diff_mean':
            empty_views.add((population, subject, level, view_kind, view_param))
    for key, score_map in view_results.items():
        level, metric, subject, view_kind, view_param = key
        if subject in published and len(score_map) < 8 and metric in ('unit_norm_diff_mean', 'mean'):
            small_views.add((population, subject, level, metric in difference, view_kind,
                             view_param))
    with_mean = {s: set(frame.loc[frame[f'mean_{s}'].notna(), 'rspo'])
                 for s in published if s in SUBJECT_KEYS}
    vv = views[(views['reference_level'] == 'voivodeship') & (views['metric'] == 'diff_mean')]
    base_at_vv = vv[vv['view_kind'] == 'base'].groupby('subject')['rspo'].apply(set)
    any_at_vv = vv.groupby('subject')['rspo'].apply(set)
    withheld_report[population] = (
        sum(len(with_mean[s] - base_at_vv.get(s, set())) for s in with_mean),
        sum(len(with_mean[s] - any_at_vv.get(s, set())) for s in with_mean))
    excluded = sorted(set(SUBJECT_KEYS) - set(published))
    print(f"{population}: {len(frame):,} rows, {frame['rspo'].nunique():,} schools; "
          f"excluded subjects: {', '.join(excluded) or 'none'}")

print('base views withheld at the voivodeship reference (difference metrics): '
      f"liceum {withheld_report['liceum'][0]} ({withheld_report['liceum'][1]} schools with no "
      f"voivodeship view), technikum {withheld_report['technikum'][0]} "
      f"({withheld_report['technikum'][1]}), all {withheld_report['all'][0]}")
small_subjects = sorted({(p, s) for p, s, *_ in small_views})
print(f'views ranking 1-7 schools: {len(small_views)}, all '
      + ', '.join(f'{p} {s}' for p, s in small_subjects))
empty_subjects = sorted({(p, s) for p, s, *_ in empty_views})
print(f'views with candidates but no ranked school: {len(empty_views)}, all '
      + ', '.join(f'{p} {s}' for p, s in empty_subjects))

liceum: 8,694 rows, 2,374 schools; excluded subjects: none


technikum: 5,102 rows, 1,770 schools; excluded subjects: historia_sztuki_r


all: 11,681 rows, 4,100 schools; excluded subjects: none
base views withheld at the voivodeship reference (difference metrics): liceum 81 (43 schools with no voivodeship view), technikum 119 (77), all 62
views ranking 1-7 schools: 11, all technikum niemiecki_r
views with candidates but no ranked school: 2, all technikum niemiecki_r


#### The figures the spec quotes

Spec §8.5: every figure the design quotes, apart from §2.2's edition comparison, is
recomputed in this notebook, so the next edition shows which of them moved. These are
the ones the cells above do not print: years per school (§3.2), the pass rate (§4.3),
the region counts behind the reference levels and the zoom ladder (§5.3, §5.7), and
what the sparse-subject rules do (§5.5).

In [14]:
# ── Spec figures not printed above (§3.2, §4.3, §5.3, §5.5, §5.7) ─────────────
from school_quality.matura import EXTENDED_SUBJECTS
from school_quality.suppression import MIN_REFERENCE_N

OWN_TYPE = ('liceum', 'technikum')
POWIAT_KEYS = sorted(p.stem for p in (GEO_DIR / 'pow').glob('*.json'))
VOIVODESHIP_YEAR = ['year', 'teryt_wojewodztwo']

# §3.2: years per school; the combined population per year.
for population in OWN_TYPE:
    years = POP_FRAMES[population].groupby('rspo')['year'].nunique().value_counts().sort_index()
    print(f'{population}, schools by number of years: '
          + ', '.join(f'{k}: {v:,}' for k, v in years.items()))
per_year = POP_FRAMES['all'].groupby('year')['rspo'].nunique()
print('all, schools per year: ' + ', '.join(f'{y} {n:,}' for y, n in per_year.items()))

# §4.3: the pass rate in 2025, the year the spec quotes.
for population in OWN_TYPE:
    frame = POP_FRAMES[population]
    rates = frame.loc[frame['year'] == 2025, 'pass_rate'].dropna()
    print(f'{population} pass rate 2025: median {rates.median():.1f}%, '
          f'{(rates >= 100).mean():.1%} at 100%')

# §5.3, §5.7: schools per region, over all years and per year; what a powiat
# reference would withhold.
for population in OWN_TYPE:
    frame = POP_FRAMES[population]
    ident = IDENTITY[IDENTITY['type'] == population]
    per_powiat = ident.groupby('teryt_powiat').size().reindex(POWIAT_KEYS, fill_value=0)
    print(f'{population}, all years: smallest voivodeship '
          f"{ident.groupby('teryt_wojewodztwo').size().min()}; of {len(POWIAT_KEYS)} powiats "
          f'{(per_powiat >= 5).sum()} with >= 5, {(per_powiat == 0).sum()} with none, '
          f'{(per_powiat == 1).sum()} with exactly one; median {per_powiat.median():g}')
    yearly = (frame.groupby(['year', 'teryt_powiat']).size().unstack(fill_value=0)
              .reindex(columns=POWIAT_KEYS, fill_value=0))
    five, one, none = ((yearly >= 5).sum(axis=1), (yearly == 1).sum(axis=1),
                       (yearly == 0).sum(axis=1))
    medians = yearly.median(axis=1)
    print(f'{population}, per year: smallest voivodeship '
          f'{frame.groupby(VOIVODESHIP_YEAR).size().min()}; powiats with >= 5 '
          f'{five.min()}-{five.max()} ({(yearly >= 5).all().sum()} in every year), exactly one '
          f'{one.min()}-{one.max()}, none {none.min()}-{none.max()}; median '
          + (f'{medians.min():g}' if medians.min() == medians.max()
             else f'{medians.min():g}-{medians.max():g}'))
    smallest = (frame.groupby(['year', 'teryt_powiat'])['rspo'].transform('size')
                .groupby(frame['rspo']).min())
    withheld = int((smallest < MIN_REFERENCE_N).sum())
    print(f'{population}: a powiat reference would withhold {withheld:,} of '
          f'{len(smallest):,} base scores ({withheld / len(smallest):.1%})')

# §5.5 a: what counting every school instead of the scored ones would publish,
# over the subjects each population publishes.
for population in POPULATIONS:
    frame = POP_FRAMES[population]
    size = frame.groupby(VOIVODESHIP_YEAR)['rspo'].transform('size')
    gained = {s: int((frame[f'mean_{s}'].notna()
                      & (frame[f'ref_n_voivodeship_{s}'] < MIN_REFERENCE_N)
                      & (size >= MIN_REFERENCE_N)).sum())
              for s in POP_SUBJECTS[population] if s in SUBJECT_KEYS}
    print(f'{population}: counting every school would add {sum(gained.values())} '
          f'voivodeship scores from references under {MIN_REFERENCE_N} scored schools: '
          + ', '.join(f'{s} {n}' for s, n in gained.items() if n))

# §5.5: (year, voivodeship) groups under the reference threshold, per subject (art
# history included for technika, where it is the reason it is not published); the
# extended base views the rule withholds; and none at the national reference.
withheld_lines, withheld_nationally = [], 0
for population in POPULATIONS:
    frame, views = POP_FRAMES[population], VIEWS[population]
    base = views[(views['metric'] == 'diff_mean') & (views['view_kind'] == 'base')]
    counts = frame.groupby(VOIVODESHIP_YEAR)[[f'mean_{s}' for s in EXTENDED_SUBJECTS]].count()
    short = (counts < MIN_REFERENCE_N).sum()
    if population in OWN_TYPE:
        print(f'{population}: (year, voivodeship) groups with < {MIN_REFERENCE_N} scored '
              f'schools, of {len(counts)}: '
              + ', '.join(f"{c.removeprefix('mean_')} {n}" for c, n in short.items() if n))
    for s in POP_SUBJECTS[population]:
        if s not in SUBJECT_KEYS:
            continue
        has = set(frame.loc[frame[f'mean_{s}'].notna(), 'rspo'])
        at = {level: set(base.loc[(base['reference_level'] == level) & (base['subject'] == s),
                                  'rspo']) for level in MATURA_LEVELS}
        withheld_nationally += len(has - at['national'])
        withheld = len(has - at['voivodeship'])
        if population in OWN_TYPE and s in EXTENDED_SUBJECTS and withheld:
            withheld_lines.append(f'{population} {s} {withheld} of {len(has)}')
print('extended base views withheld at the voivodeship reference: '
      + ', '.join(withheld_lines))
print(f'base views withheld at the national reference: {withheld_nationally}')

# §5.5 e: published extended subjects' own year counts against their schools'.
pairs = fewer = single = 0
for population in OWN_TYPE:
    frame = POP_FRAMES[population]
    school_years = frame.groupby('rspo')['year'].nunique()
    for s in (s for s in POP_SUBJECTS[population] if s in EXTENDED_SUBJECTS):
        subject_years = frame[frame[f'mean_{s}'].notna()].groupby('rspo')['year'].nunique()
        of_school = school_years.loc[subject_years.index]
        pairs += len(subject_years)
        fewer += int((subject_years < of_school).sum())
        single += int(((subject_years == 1) & (of_school >= 3)).sum())
print(f'(school, extended subject) pairs: {pairs:,}; {fewer:,} ({fewer / pairs:.1%}) with '
      f'fewer subject-years than school-years; {single:,} on a single subject-year at a '
      f'school with >= 3 years')

liceum, schools by number of years: 1: 146, 2: 102, 3: 160, 4: 1,966
technikum, schools by number of years: 1: 62, 2: 84, 3: 1,624
all, schools per year: 2024 3,835, 2025 3,863, 2026 3,983
liceum pass rate 2025: median 95.8%, 29.6% at 100%
technikum pass rate 2025: median 78.3%, 3.3% at 100%
liceum, all years: smallest voivodeship 48; of 380 powiats 158 with >= 5, 3 with none, 36 with exactly one; median 4
liceum, per year: smallest voivodeship 40; powiats with >= 5 141-150 (126 in every year), exactly one 43-50, none 3-4; median 3-4
liceum: a powiat reference would withhold 684 of 2,374 base scores (28.8%)
technikum, all years: smallest voivodeship 49; of 380 powiats 131 with >= 5, 2 with none, 34 with exactly one; median 4
technikum, per year: smallest voivodeship 47; powiats with >= 5 123-126 (118 in every year), exactly one 34-42, none 2-3; median 3
technikum: a powiat reference would withhold 701 of 1,770 base scores (39.6%)
liceum: counting every school would add 111 voivodeship 

technikum: (year, voivodeship) groups with < 5 scored schools, of 48: chemia_r 21, fizyka_r 5, historia_r 18, informatyka_r 1, historia_sztuki_r 48, niemiecki_r 46


extended base views withheld at the voivodeship reference: liceum historia_sztuki_r 30 of 295, liceum niemiecki_r 51 of 266, technikum chemia_r 40 of 175, technikum fizyka_r 18 of 333, technikum historia_r 43 of 258, technikum informatyka_r 3 of 361, technikum niemiecki_r 15 of 21
base views withheld at the national reference: 0
(school, extended subject) pairs: 21,200; 8,384 (39.5%) with fewer subject-years than school-years; 3,672 on a single subject-year at a school with >= 3 years


### 6b. scale.json and schools-index.json

`scale.json` holds what turns a score into a colour: σ, centre, p1 and p99 per (level,
metric, subject), computed on this population's base scores — the same rules as E8
(centre 0 for the difference metrics, the empirical mean for `mean`, `median` and
`composite_min`; p1/p99 on scores rounded to 4 places, as the shards publish them).
Every entry must have at least 2 scores and σ > 0: a missing or `null` entry would
send the frontend's `scaleFor` to its `{sigma: 1, centre: 0}` fallback and paint every
school saturated.

The value-filter slider spans **every subject of the population**, not
`composite_min` alone: its default position is its minimum, and a composite-only
minimum would hide every school whose extended score sits below it.

`schools-index.json` is identity plus what the map needs before any shard loads: the
type, the pass rate per year (published after the August resit, under the September
pin), and, for extended subjects only, how many years a subject has a published mean
(`subject_years`) and its median takers per year (`subject_takers`).

In [15]:
# ── Coordinates (optional) and scale.json per population ─────────────────────
from datetime import datetime

import numpy as np

from export_io import commit_export, json_payloads_equal, write_json
from school_quality.matura import COMPOSITE, POPULATIONS, subject_groups

COORDS = {}
if COORDS_PATH.exists():
    coords_df = pd.read_csv(COORDS_PATH)
    for rspo, lat, lon in zip(coords_df['rspo'], coords_df['latitude'], coords_df['longitude'],
                              strict=True):
        if pd.notna(lat) and pd.notna(lon):
            COORDS[int(rspo)] = (float(lat), float(lon))
print(f'Loaded coordinates for {len(COORDS):,} schools from {COORDS_PATH.name}; '
      f'{len(set(IDENTITY["rspo"]) & set(COORDS)):,} of the {len(IDENTITY):,} matura schools '
      f'have one')

for population in POPULATIONS:
    views, subjects = VIEWS[population], POP_SUBJECTS[population]
    base = views[views['view_kind'] == 'base']
    out_dir = DOCS_DATA_DIR / population
    out_dir.mkdir(parents=True, exist_ok=True)

    school = {}
    for level in MATURA_LEVELS:
        school[level] = {}
        for metric in EXPORT_METRICS:
            at = base[(base['metric'] == metric)
                      & (base['reference_level'] == SOURCE_LEVEL(metric, level))]
            school[level][metric] = {}
            for subject in subjects:
                scores = at.loc[at['subject'] == subject, 'score']
                sigma = float(scores.std())
                assert len(scores) >= 2 and np.isfinite(sigma) and sigma > 0, (
                    population, level, metric, subject, len(scores))
                centre = (float(scores.mean())
                          if subject == COMPOSITE or metric in ('mean', 'median') else 0.0)
                rounded = sorted(round(float(v), 4) for v in scores)
                p1 = rounded[max(0, int(0.01 * len(rounded)) - 1)]
                p99 = rounded[min(len(rounded) - 1, int(0.99 * len(rounded)))]
                school[level][metric][subject] = {
                    'sigma': round(sigma, 4), 'sigma_centre': round(centre, 4),
                    'p1': round(p1, 4), 'p99': round(p99, 4)}

    slider = {}
    for level in MATURA_LEVELS:
        slider[level] = {}
        for metric in EXPORT_METRICS:
            at = base[(base['metric'] == metric)
                      & (base['reference_level'] == SOURCE_LEVEL(metric, level))
                      & base['subject'].isin(subjects)]
            lo, hi = float(at['score'].min()), float(at['score'].max())
            span = hi - lo
            slider[level][metric] = {
                'min': round(lo, 4), 'max': round(hi, 4),
                'p1': round(float(at['score'].quantile(0.01)), 4),
                'p99': round(float(at['score'].quantile(0.99)), 4),
                'step': round(span / 100, 4) if span > 0 else 0.01}
            assert (at.groupby('subject')['score'].min() >= lo).all(), (population, metric)

    # The schools a composite-only range would have hidden (spec §6.3).
    at = base[(base['metric'] == 'mean') & (base['reference_level'] == PRIMARY_REFERENCE_LEVEL)]
    composite_floor = at.loc[at['subject'] == COMPOSITE, 'score'].min()
    below = int((at.loc[at['subject'] == 'matematyka_r', 'score'] < composite_floor).sum())
    print(f"{population}: mean slider minimum {slider['voivodeship']['mean']['min']:.2f}, "
          f'composite minimum {composite_floor:.2f}; {below} schools with matematyka_r below '
          f'the composite minimum stay visible')

    years = sorted(int(y) for y in POP_FRAMES[population]['year'].unique())
    payload = {
        'metadata': {
            'generated_at': datetime.now().isoformat(timespec='seconds'),
            'population': population,
            'default_metric': PRIMARY_METRIC,
            'metrics': list(EXPORT_METRICS),
            'levels': list(MATURA_LEVELS),
            'subjects': list(subjects),
            'subject_groups': subject_groups(population),
            'years_in_data': years,
            'slider_ranges': slider,
            'n_ranked': N_RANKED[population],
        },
        'school': school,
    }
    scale_path = out_dir / 'scale.json'
    commit_export(
        scale_path,
        unchanged=(not FORCE_REGENERATE) and json_payloads_equal(payload, scale_path),
        write=lambda p=scale_path, pay=payload: write_json(p, pay),
        detail=f'{population}: {len(MATURA_LEVELS)} levels × {len(EXPORT_METRICS)} metrics '
               f'× {len(subjects)} subjects',
        force_regenerate=FORCE_REGENERATE,
        write_output_files=WRITE_OUTPUT_FILES,
    )

Loaded coordinates for 16,992 schools from school_coords.csv; 4,140 of the 4,144 matura schools have one


liceum: mean slider minimum 0.00, composite minimum 15.40; 262 schools with matematyka_r below the composite minimum stay visible
  scale.json: no changes, skipping
technikum: mean slider minimum 0.00, composite minimum 12.61; 628 schools with matematyka_r below the composite minimum stay visible


  scale.json: no changes, skipping


all: mean slider minimum 0.00, composite minimum 12.61; 824 schools with matematyka_r below the composite minimum stay visible
  scale.json: no changes, skipping


In [16]:
# ── schools-index.json per population ────────────────────────────────────────
SUBJECT_YEARS, SUBJECT_TAKERS = {}, {}
for population in POPULATIONS:
    frame = POP_FRAMES[population]
    extended = subject_groups(population)['extended']
    members = sorted(int(r) for r in frame['rspo'].unique())
    ident = IDENTITY.set_index('rspo').loc[members]
    by_school = frame.sort_values(['rspo', 'year']).groupby('rspo')
    n_years = by_school['year'].nunique()
    pass_rate = {
        int(rspo): {str(int(y)): (None if pd.isna(v) else round(float(v), 4))
                    for y, v in zip(group['year'], group['pass_rate'], strict=True)}
        for rspo, group in by_school}
    nulls = sum(v is None for rates in pass_rate.values() for v in rates.values())
    assert nulls == int(frame['pass_rate'].isna().sum()), (population, nulls)

    subject_years, subject_takers = {}, {}
    for s in extended:
        has = frame[frame[f'mean_{s}'].notna()]
        for rspo, group in has.groupby('rspo'):
            subject_years.setdefault(int(rspo), {})[s] = int(group['year'].nunique())
            subject_takers.setdefault(int(rspo), {})[s] = int(round(float(np.median(group[f'n_{s}']))))
    # Review focus 3: a subject's years are its own, never more than the school's.
    assert all(v <= n_years[r] for r, by_s in subject_years.items() for v in by_s.values())
    SUBJECT_YEARS[population], SUBJECT_TAKERS[population] = subject_years, subject_takers

    schools = {
        'rspo':        members,
        'name':        [str(v) for v in ident['school_name']],
        'teryt':       [str(v) for v in ident['teryt']],
        'powiat':      [str(v) for v in ident['teryt_powiat']],
        'miejscowosc': [str(v) for v in ident['miejscowosc']],
        'ulica_nr':    [str(v) for v in ident['ulica_nr']],
        'is_public':   [str(v) for v in ident['is_public']],
        'n_years':     [int(n_years[r]) for r in members],
        'lat':         [COORDS.get(r, (None, None))[0] for r in members],
        'lon':         [COORDS.get(r, (None, None))[1] for r in members],
    }
    schools['on_map'] = [v is not None for v in schools['lat']]
    schools['type'] = [str(v) for v in ident['type']]
    schools['pass_rate'] = [pass_rate[r] for r in members]
    schools['subject_years'] = [subject_years.get(r, {}) for r in members]
    schools['subject_takers'] = [subject_takers.get(r, {}) for r in members]
    index_payload = {'metadata': {'generated_at': datetime.now().isoformat(timespec='seconds'),
                                  'population': population},
                     'schools': schools}
    index_path = DOCS_DATA_DIR / population / 'schools-index.json'
    commit_export(
        index_path,
        unchanged=(not FORCE_REGENERATE) and json_payloads_equal(index_payload, index_path),
        write=lambda p=index_path, pay=index_payload: write_json(p, pay),
        detail=f'{population}: {len(members):,} schools',
        force_regenerate=FORCE_REGENERATE,
        write_output_files=WRITE_OUTPUT_FILES,
    )
    print(f"{population}: {len(members):,} schools, {sum(schools['on_map']):,} on the map, "
          f'{nulls} pass rates null')

  schools-index.json: no changes, skipping
liceum: 2,374 schools, 2,371 on the map, 4 pass rates null


  schools-index.json: no changes, skipping
technikum: 1,770 schools, 1,769 on the map, 3 pass rates null


  schools-index.json: no changes, skipping
all: 4,100 schools, 4,096 on the map, 5 pass rates null


### 6c. Region aggregates

What the choropleth draws: voivodeships scored nationally, powiats scored within their
voivodeship (a region is scored against the level above it — scoring it against itself
puts every region at zero). Row identity comes from the polygons: a powiat with no
school of the population is a real row with `n_schools = 0`.

Sparse subjects change three things (spec §5.5 b–c, §6.4): a region's score for a
subject weights each school by **that subject's** takers; a difference-metric score
needs at least 5 schools **with a published mean in that subject** in the parent
region; and the region files say how many schools and takers stand behind each subject
(`n_scored`, `n_takers`), so the map does not show 17 schools and 1,127 pupils beside a
score that rests on one school's 21 takers.

In [17]:
# ── Region aggregates per population: voivodeship and powiat ─────────────────
import json
from datetime import datetime

import numpy as np

from export_io import commit_export, json_payloads_equal, write_json
from school_quality.aggregate import (
    aggregate_region,
    rank_and_percentile,
    region_score_publishable,
    sibling_percentiles,
    weighting_for,
)
from school_quality.matura import BASIC_KEYS, COMPOSITE, POPULATIONS, region_display_name
from school_quality.suppression import MIN_PERCENTILE_N, MIN_REFERENCE_N

REGION_LEVELS = {'voivodeship': (2, None), 'powiat': (4, 2)}
PARENT_REFERENCE_LEVEL = {'voivodeship': 'national', 'powiat': 'voivodeship'}
NAME_COLUMN = {2: 'wojewodztwo', 4: 'powiat'}


def _bbox_centre(geometry):
    """(lat, lon) at the centre of a GeoJSON geometry's bounding box."""
    xs, ys = [], []

    def walk(node):
        if node and isinstance(node[0], (int, float)):
            xs.append(node[0])
            ys.append(node[1])
            return
        for child in node:
            walk(child)

    walk(geometry['coordinates'])
    return round((min(ys) + max(ys)) / 2, 5), round((min(xs) + max(xs)) / 2, 5)


def geometry_regions(level, width):
    """{teryt: {'lat', 'lon', 'name'}} for every polygon at `level` (E8 cell 75)."""
    files = {
        'voivodeship': [GEO_DIR / 'kraj.json'],
        'powiat': sorted((GEO_DIR / 'woj').glob('*.json')),
    }[level]
    out = {}
    for f in files:
        for feature in json.loads(f.read_text(encoding='utf-8'))['features']:
            key = feature['properties']['JPT_KOD_JE'][:width]
            if key in out:
                continue
            lat, lon = _bbox_centre(feature['geometry'])
            out[key] = {'lat': lat, 'lon': lon, 'name': feature['properties']['name']}
    return out


for population in POPULATIONS:
    frame, views, subjects = POP_FRAMES[population], VIEWS[population], POP_SUBJECTS[population]
    base = views[views['view_kind'] == 'base']
    members_all = sorted(int(r) for r in frame['rspo'].unique())
    ident = IDENTITY.set_index('rspo').loc[members_all]
    teryt_by_rspo = ident['teryt'].astype(str).to_dict()
    years = sorted(int(y) for y in frame['year'].unique())

    # Rule b: each subject's median takers per year over the school's years with a
    # published mean; composite_min takes the largest of the three basic subjects'.
    takers = {}
    for s in subjects:
        if s == COMPOSITE:
            continue
        has = frame[frame[f'mean_{s}'].notna()]
        takers[s] = {int(r): int(round(float(np.median(g[f'n_{s}']))))
                     for r, g in has.groupby('rspo')}
    takers[COMPOSITE] = {r: max(takers[s][r] for s in BASIC_KEYS) for r in members_all}

    for level, (width, parent_width) in REGION_LEVELS.items():
        ref_level = PARENT_REFERENCE_LEVEL[level]
        geo_rows = geometry_regions(level, width)
        regions = sorted(geo_rows)
        members = {}
        for rspo, teryt in teryt_by_rspo.items():
            members.setdefault(teryt[:width], []).append(rspo)
        orphans = set(members) - set(geo_rows)
        assert not orphans, f'{population}: {level} keys with no polygon: {sorted(orphans)[:5]}'

        def parent_of(region):
            return region[:parent_width] if parent_width else ''

        siblings = {}
        for region in regions:
            siblings[parent_of(region)] = siblings.get(parent_of(region), 0) + 1
        n_scored = {s: [sum(1 for x in members.get(r, []) if x in takers[s]) for r in regions]
                    for s in subjects}
        n_takers = {s: [int(sum(takers[s].get(x, 0) for x in members.get(r, []))) for r in regions]
                    for s in subjects}
        parent_scored = {s: {} for s in subjects}
        for s in subjects:
            for region, n in zip(regions, n_scored[s], strict=True):
                parent_scored[s][parent_of(region)] = parent_scored[s].get(parent_of(region), 0) + n

        name_col = NAME_COLUMN[width]
        data_names = {r: str(ident.loc[rspos[0], name_col]) for r, rspos in members.items()}
        payload_regions = {
            'teryt': regions,
            'name': [region_display_name(data_names.get(r) or geo_rows[r]['name']) for r in regions],
            'parent': [parent_of(r) for r in regions],
            'lat': [geo_rows[r]['lat'] for r in regions],
            'lon': [geo_rows[r]['lon'] for r in regions],
            'n_schools': [len(members.get(r, [])) for r in regions],
            'n_students': [int(sum(takers[COMPOSITE][x] for x in members.get(r, [])))
                           for r in regions],
            'n_scored': n_scored,
            'n_takers': n_takers,
            'score': {}, 'rank': {}, 'pct': {}, 'n_ranked': {},
        }
        for metric in EXPORT_METRICS:
            stat = weighting_for(metric)
            for block in ('score', 'rank', 'pct', 'n_ranked'):
                payload_regions[block][metric] = {}
            wanted = SOURCE_LEVEL(metric, ref_level)
            for subject in subjects:
                cells = base[(base['metric'] == metric) & (base['subject'] == subject)
                             & (base['reference_level'] == wanted)]
                score_by_rspo = dict(zip(cells['rspo'], cells['score'], strict=True))
                raw = {r: aggregate_region([score_by_rspo.get(x) for x in members.get(r, [])],
                                           [takers[subject].get(x, 0) for x in members.get(r, [])],
                                           stat)
                       for r in regions}
                # Rule c: the parent's schools WITH A PUBLISHED MEAN in this subject.
                allowed = {r: region_score_publishable(metric, parent_scored[subject][parent_of(r)],
                                                       siblings[parent_of(r)])
                           for r in regions}
                scores = [None if not allowed[r] or raw[r] is None else round(float(raw[r]), 4)
                          for r in regions]
                ranks, _ = rank_and_percentile(scores)   # national, every region at this level
                pcts = [None] * len(regions)
                by_parent = {}
                for i, r in enumerate(regions):
                    by_parent.setdefault(parent_of(r), []).append(i)
                for idxs in by_parent.values():           # among siblings with a score
                    for i, pc in zip(idxs, sibling_percentiles([scores[i] for i in idxs]),
                                     strict=True):
                        pcts[i] = pc
                payload_regions['score'][metric][subject] = scores
                payload_regions['rank'][metric][subject] = ranks
                payload_regions['pct'][metric][subject] = pcts
                payload_regions['n_ranked'][metric][subject] = sum(v is not None for v in scores)

        sigma_r, centre_r = {}, {}
        for metric in EXPORT_METRICS:
            sigma_r[metric], centre_r[metric] = {}, {}
            for subject in subjects:
                vals = [v for v in payload_regions['score'][metric][subject] if v is not None]
                series = pd.Series(vals, dtype=float)
                sd = float(series.std()) if len(vals) > 1 else 0.0
                sigma_r[metric][subject] = round(0.0 if np.isnan(sd) else sd, 4)
                centre_r[metric][subject] = round(float(series.mean()) if vals else 0.0, 4)

        payload = {
            'metadata': {
                'generated_at': datetime.now().isoformat(timespec='seconds'),
                'population': population,
                'level': level,
                'metrics': list(EXPORT_METRICS),
                'subjects': list(subjects),
                'years_in_data': years,
                'sigma': sigma_r,
                'sigma_centre': centre_r,
                'min_reference_n': MIN_REFERENCE_N,
                'min_percentile_n': MIN_PERCENTILE_N,
            },
            'regions': payload_regions,
        }
        out_path = DOCS_DATA_DIR / population / f'regions-{level}.json'
        commit_export(
            out_path,
            unchanged=(not FORCE_REGENERATE) and json_payloads_equal(payload, out_path),
            write=lambda p=out_path, pay=payload: write_json(p, pay),
            detail=f'{population}: {len(regions):,} {level} regions',
            force_regenerate=FORCE_REGENERATE,
            write_output_files=WRITE_OUTPUT_FILES,
        )

  regions-voivodeship.json: no changes, skipping


  regions-powiat.json: wrote 408.7 KB, liceum: 380 powiat regions


  regions-voivodeship.json: no changes, skipping


  regions-powiat.json: wrote 376.8 KB, technikum: 380 powiat regions


  regions-voivodeship.json: no changes, skipping


  regions-powiat.json: wrote 411.9 KB, all: 380 powiat regions


### 6d. Shards and analyst xlsx

One shard per (powiat, metric) holds every view of that powiat's schools, at every
level the metric has: both levels for the difference metrics, the primary level alone
for `mean`/`median`, whose score has no reference population. Every powiat gets a file,
empty or not, because the map loads a voivodeship's shards together and one missing
file would fail them all. A school carries only the extended subjects it has a view
in; a missing subject can mean *withheld*, which is why year counts come from
`subject_years` in the index, never from shard keys (spec §6.5).

The analyst xlsx is per population (spec §6.6): one file for all three would cross
Excel's row ceiling within three annual updates.

In [18]:
# ── Shards: docs/data/matura/<population>/powiat/{teryt4}-{metric}.json ──────
from datetime import datetime

from export_io import commit_export, json_payloads_equal, write_json
from school_quality.matura import POPULATIONS
from school_quality.views import cell_payload, view_param_key

POWIATS = sorted(p.stem for p in (GEO_DIR / 'pow').glob('*.json'))
assert len(POWIATS) == 380, len(POWIATS)


def build_metric_payload(views, metric, level):
    """{rspo: {subject: {view_kind: cell or {param: cell}}}} for one metric at one level."""
    at = views[(views['metric'] == metric) & (views['reference_level'] == level)]
    assert not at.empty, (metric, level)
    by_school = {}
    for row in at.itertuples(index=False):
        subject_views = by_school.setdefault(int(row.rspo), {}).setdefault(row.subject, {})
        cell = cell_payload(row.score, row.rank_overall, row.pct_overall)
        if row.view_kind == 'base':
            subject_views['base'] = cell
        else:
            subject_views.setdefault(row.view_kind, {})[
                view_param_key(row.view_kind, row.view_param)] = cell
    return by_school


powiat_of = IDENTITY.set_index('rspo')['teryt_powiat'].astype(str).to_dict()
for population in POPULATIONS:
    views = VIEWS[population]
    shard_dir = DOCS_DATA_DIR / population / 'powiat'
    shard_dir.mkdir(parents=True, exist_ok=True)
    for metric in EXPORT_METRICS:
        shard_levels = ([PRIMARY_REFERENCE_LEVEL] if metric in ('mean', 'median')
                        else list(MATURA_LEVELS))
        per_level = {level: build_metric_payload(views, metric, level) for level in shard_levels}
        schools = set().union(*per_level.values())
        for level, payload in per_level.items():
            missing = schools - set(payload)
            assert not missing, (
                f'{population} {metric}: {len(missing)} schools without a {level} record')
        by_powiat = {key: {} for key in POWIATS}
        for rspo in sorted(schools):
            key = powiat_of[rspo]
            assert key in by_powiat, f'{population}: school {rspo} sits in powiat {key}, no polygon'
            by_powiat[key][str(rspo)] = {level: per_level[level][rspo] for level in shard_levels}
        for powiat in POWIATS:
            payload = {
                'metadata': {
                    'generated_at': datetime.now().isoformat(timespec='seconds'),
                    'population': population,
                    'metric': metric,
                    'powiat': powiat,
                    'levels': shard_levels,
                },
                'schools': by_powiat[powiat],
            }
            out_path = shard_dir / f'{powiat}-{metric}.json'
            commit_export(
                out_path,
                unchanged=(not FORCE_REGENERATE) and json_payloads_equal(payload, out_path),
                write=lambda p=out_path, pay=payload: write_json(p, pay),
                detail=f'{len(by_powiat[powiat])} schools',
                force_regenerate=FORCE_REGENERATE,
                write_output_files=WRITE_OUTPUT_FILES,
            )

  0201-mean.json: no changes, skipping
  0202-mean.json: no changes, skipping
  0203-mean.json: no changes, skipping
  0204-mean.json: no changes, skipping
  0205-mean.json: no changes, skipping
  0206-mean.json: no changes, skipping
  0207-mean.json: no changes, skipping
  0208-mean.json: no changes, skipping
  0209-mean.json: no changes, skipping
  0210-mean.json: no changes, skipping
  0211-mean.json: no changes, skipping
  0212-mean.json: no changes, skipping
  0213-mean.json: no changes, skipping
  0214-mean.json: no changes, skipping
  0215-mean.json: no changes, skipping
  0216-mean.json: no changes, skipping
  0217-mean.json: no changes, skipping
  0218-mean.json: no changes, skipping
  0219-mean.json: no changes, skipping
  0220-mean.json: no changes, skipping
  0221-mean.json: no changes, skipping
  0222-mean.json: no changes, skipping
  0223-mean.json: no changes, skipping
  0224-mean.json: no changes, skipping
  0225-mean.json: no changes, skipping
  0226-mean.json: no chan

  1419-mean.json: no changes, skipping
  1420-mean.json: no changes, skipping
  1421-mean.json: no changes, skipping
  1422-mean.json: no changes, skipping
  1423-mean.json: no changes, skipping
  1424-mean.json: no changes, skipping
  1425-mean.json: no changes, skipping
  1426-mean.json: no changes, skipping
  1427-mean.json: no changes, skipping
  1428-mean.json: no changes, skipping
  1429-mean.json: no changes, skipping
  1430-mean.json: no changes, skipping
  1432-mean.json: no changes, skipping
  1433-mean.json: no changes, skipping
  1434-mean.json: no changes, skipping
  1435-mean.json: no changes, skipping
  1436-mean.json: no changes, skipping
  1437-mean.json: no changes, skipping
  1438-mean.json: no changes, skipping
  1461-mean.json: no changes, skipping
  1462-mean.json: no changes, skipping
  1463-mean.json: no changes, skipping
  1464-mean.json: no changes, skipping
  1465-mean.json: no changes, skipping
  1601-mean.json: no changes, skipping
  1602-mean.json: no chan

  2610-mean.json: no changes, skipping
  2611-mean.json: no changes, skipping
  2612-mean.json: no changes, skipping
  2613-mean.json: no changes, skipping
  2661-mean.json: no changes, skipping
  2801-mean.json: no changes, skipping
  2802-mean.json: no changes, skipping
  2803-mean.json: no changes, skipping
  2804-mean.json: no changes, skipping
  2805-mean.json: no changes, skipping
  2806-mean.json: no changes, skipping
  2807-mean.json: no changes, skipping
  2808-mean.json: no changes, skipping
  2809-mean.json: no changes, skipping
  2810-mean.json: no changes, skipping
  2811-mean.json: no changes, skipping
  2812-mean.json: no changes, skipping
  2813-mean.json: no changes, skipping
  2814-mean.json: no changes, skipping
  2815-mean.json: no changes, skipping
  2816-mean.json: no changes, skipping
  2817-mean.json: no changes, skipping
  2818-mean.json: no changes, skipping
  2819-mean.json: no changes, skipping
  2861-mean.json: no changes, skipping
  2862-mean.json: no chan

  0201-median.json: no changes, skipping
  0202-median.json: no changes, skipping
  0203-median.json: no changes, skipping
  0204-median.json: no changes, skipping
  0205-median.json: no changes, skipping
  0206-median.json: no changes, skipping
  0207-median.json: no changes, skipping
  0208-median.json: no changes, skipping
  0209-median.json: no changes, skipping
  0210-median.json: no changes, skipping
  0211-median.json: no changes, skipping
  0212-median.json: no changes, skipping
  0213-median.json: no changes, skipping
  0214-median.json: no changes, skipping
  0215-median.json: no changes, skipping
  0216-median.json: no changes, skipping
  0217-median.json: no changes, skipping
  0218-median.json: no changes, skipping
  0219-median.json: no changes, skipping
  0220-median.json: no changes, skipping
  0221-median.json: no changes, skipping
  0222-median.json: no changes, skipping
  0223-median.json: no changes, skipping
  0224-median.json: no changes, skipping
  0225-median.js

  1611-median.json: no changes, skipping
  1661-median.json: no changes, skipping
  1801-median.json: no changes, skipping
  1802-median.json: no changes, skipping
  1803-median.json: no changes, skipping
  1804-median.json: no changes, skipping
  1805-median.json: no changes, skipping
  1806-median.json: no changes, skipping
  1807-median.json: no changes, skipping
  1808-median.json: no changes, skipping
  1809-median.json: no changes, skipping
  1810-median.json: no changes, skipping
  1811-median.json: no changes, skipping
  1812-median.json: no changes, skipping
  1813-median.json: no changes, skipping
  1814-median.json: no changes, skipping
  1815-median.json: no changes, skipping
  1816-median.json: no changes, skipping
  1817-median.json: no changes, skipping
  1818-median.json: no changes, skipping
  1819-median.json: no changes, skipping
  1820-median.json: no changes, skipping
  1821-median.json: no changes, skipping
  1861-median.json: no changes, skipping
  1862-median.js

...

  3212-diff_mean.json: no changes, skipping
  3213-diff_mean.json: no changes, skipping
  3214-diff_mean.json: no changes, skipping
  3215-diff_mean.json: no changes, skipping
  3216-diff_mean.json: no changes, skipping
  3217-diff_mean.json: no changes, skipping
  3218-diff_mean.json: no changes, skipping
  3261-diff_mean.json: no changes, skipping
  3262-diff_mean.json: no changes, skipping
  3263-diff_mean.json: no changes, skipping


  0201-unit_norm_diff_mean.json: no changes, skipping
  0202-unit_norm_diff_mean.json: no changes, skipping
  0203-unit_norm_diff_mean.json: no changes, skipping
  0204-unit_norm_diff_mean.json: no changes, skipping
  0205-unit_norm_diff_mean.json: no changes, skipping
  0206-unit_norm_diff_mean.json: no changes, skipping
  0207-unit_norm_diff_mean.json: no changes, skipping
  0208-unit_norm_diff_mean.json: no changes, skipping
  0209-unit_norm_diff_mean.json: no changes, skipping
  0210-unit_norm_diff_mean.json: no changes, skipping
  0211-unit_norm_diff_mean.json: no changes, skipping
  0212-unit_norm_diff_mean.json: no changes, skipping
  0213-unit_norm_diff_mean.json: no changes, skipping
  0214-unit_norm_diff_mean.json: no changes, skipping
  0215-unit_norm_diff_mean.json: no changes, skipping
  0216-unit_norm_diff_mean.json: no changes, skipping
  0217-unit_norm_diff_mean.json: no changes, skipping
  0218-unit_norm_diff_mean.json: no changes, skipping
  0219-unit_norm_diff_mean.j

  1061-unit_norm_diff_mean.json: no changes, skipping
  1062-unit_norm_diff_mean.json: no changes, skipping
  1063-unit_norm_diff_mean.json: no changes, skipping
  1201-unit_norm_diff_mean.json: no changes, skipping
  1202-unit_norm_diff_mean.json: no changes, skipping
  1203-unit_norm_diff_mean.json: no changes, skipping
  1204-unit_norm_diff_mean.json: no changes, skipping
  1205-unit_norm_diff_mean.json: no changes, skipping
  1206-unit_norm_diff_mean.json: no changes, skipping
  1207-unit_norm_diff_mean.json: no changes, skipping
  1208-unit_norm_diff_mean.json: no changes, skipping
  1209-unit_norm_diff_mean.json: no changes, skipping
  1210-unit_norm_diff_mean.json: no changes, skipping
  1211-unit_norm_diff_mean.json: no changes, skipping
  1212-unit_norm_diff_mean.json: no changes, skipping
  1213-unit_norm_diff_mean.json: no changes, skipping
  1214-unit_norm_diff_mean.json: no changes, skipping
  1215-unit_norm_diff_mean.json: no changes, skipping
  1216-unit_norm_diff_mean.j

  1806-unit_norm_diff_mean.json: no changes, skipping
  1807-unit_norm_diff_mean.json: no changes, skipping
  1808-unit_norm_diff_mean.json: no changes, skipping
  1809-unit_norm_diff_mean.json: no changes, skipping
  1810-unit_norm_diff_mean.json: no changes, skipping
  1811-unit_norm_diff_mean.json: no changes, skipping
  1812-unit_norm_diff_mean.json: no changes, skipping
  1813-unit_norm_diff_mean.json: no changes, skipping
  1814-unit_norm_diff_mean.json: no changes, skipping
  1815-unit_norm_diff_mean.json: no changes, skipping
  1816-unit_norm_diff_mean.json: no changes, skipping
  1817-unit_norm_diff_mean.json: no changes, skipping
  1818-unit_norm_diff_mean.json: no changes, skipping
  1819-unit_norm_diff_mean.json: no changes, skipping
  1820-unit_norm_diff_mean.json: no changes, skipping
  1821-unit_norm_diff_mean.json: no changes, skipping
  1861-unit_norm_diff_mean.json: no changes, skipping
  1862-unit_norm_diff_mean.json: no changes, skipping
  1863-unit_norm_diff_mean.j

  2604-unit_norm_diff_mean.json: no changes, skipping
  2605-unit_norm_diff_mean.json: no changes, skipping
  2606-unit_norm_diff_mean.json: no changes, skipping
  2607-unit_norm_diff_mean.json: no changes, skipping
  2608-unit_norm_diff_mean.json: no changes, skipping
  2609-unit_norm_diff_mean.json: no changes, skipping
  2610-unit_norm_diff_mean.json: no changes, skipping
  2611-unit_norm_diff_mean.json: no changes, skipping
  2612-unit_norm_diff_mean.json: no changes, skipping
  2613-unit_norm_diff_mean.json: no changes, skipping
  2661-unit_norm_diff_mean.json: no changes, skipping
  2801-unit_norm_diff_mean.json: no changes, skipping
  2802-unit_norm_diff_mean.json: no changes, skipping
  2803-unit_norm_diff_mean.json: no changes, skipping
  2804-unit_norm_diff_mean.json: no changes, skipping
  2805-unit_norm_diff_mean.json: no changes, skipping
  2806-unit_norm_diff_mean.json: no changes, skipping
  2807-unit_norm_diff_mean.json: no changes, skipping
  2808-unit_norm_diff_mean.j

In [19]:
# ── Analyst xlsx: output/matura/<population>/schools-{metric}.xlsx ───────────
from export_io import write_xlsx, xlsx_unchanged
from school_quality.matura import BASIC_SUBJECTS, EXTENDED_SUBJECTS

EXPECTED_XLSX_COLUMNS = [
    'rspo', 'school_name', 'school_type', 'miejscowosc', 'ulica_nr',
    'powiat', 'gmina', 'typ_gminy', 'is_public', 'n_years',
    'metric', 'subject', 'view_kind', 'view_param',
    'score', 'rank_overall', 'pct_overall', 'n_in_view', 'n_students',
]
POPULATION_NOTE = {
    'liceum': 'licea (youth and art licea), each compared with licea; 2023 onward',
    'technikum': 'technika, each compared with technika; 2024 onward',
    'all': 'licea and technika together, compared with every secondary school; 2024 onward',
}


def build_long_frame(population, metric):
    at = VIEWS[population]
    at = at[(at['metric'] == metric) & (at['reference_level'] == PRIMARY_REFERENCE_LEVEL)]
    n_years = (POP_FRAMES[population].groupby('rspo')['year'].nunique()
               .rename('n_years').reset_index())
    merged = (at.merge(IDENTITY.rename(columns={'type': 'school_type'}), on='rspo', how='left',
                       validate='many_to_one')
              .merge(n_years, on='rspo', how='left', validate='many_to_one'))
    merged['view_param'] = [view_param_key(kind, param) or ''
                            for kind, param in zip(merged['view_kind'], merged['view_param'],
                                                   strict=True)]
    merged = (merged[EXPECTED_XLSX_COLUMNS]
              .sort_values(['rspo', 'subject', 'view_kind', 'view_param'])
              .reset_index(drop=True))
    # Excel's sheet limit minus the header row: fail loudly rather than truncate.
    assert len(merged) <= 1_048_575, (population, metric, len(merged))
    return merged


def build_legend(population, metric):
    aggregation = ('arithmetic mean across years (equal weight per year)'
                   if metric in ('mean', 'median')
                   else "weighted mean across years (weighted by the subject's takers)")
    subject_rows = [(key, source) for key, (_, source) in BASIC_SUBJECTS.items()]
    subject_rows += [(key, source) for key, (_, source) in EXTENDED_SUBJECTS.items()
                     if key in POP_SUBJECTS[population]]
    rows = [
        ('FILE', f'{population}/schools-{metric}.xlsx'),
        ('population', POPULATION_NOTE[population]),
        ('metric', metric),
        ('aggregation across years', aggregation),
        ('reference level', f'{PRIMARY_REFERENCE_LEVEL}, within the population '
                            '(the map also offers the national level)'),
        ('', ''),
        ('COLUMN', 'MEANING'),
        ('rspo', 'Unique school identifier (stable across years)'),
        ('school_name', 'School name'),
        ('school_type', 'liceum or technikum'),
        ('miejscowosc', 'Town/locality'),
        ('ulica_nr', 'Street and number'),
        ('powiat', 'County (administrative)'),
        ('gmina', 'Municipality (administrative)'),
        ('typ_gminy', 'Municipality type (urban/rural/urban-rural)'),
        ('is_public', 'Tak = public school, Nie = private/non-public'),
        ('n_years', 'Number of years the school has data in this population'),
        ('metric', 'How the per-year subject score is computed (see above)'),
        ('subject', 'see SUBJECT below'),
        ('view_kind', 'base / loo / single_year / last_k (see below)'),
        ('view_param', 'For loo: the excluded year. For single_year: the year. '
                       'For last_k: k (number of most recent years). Empty for base.'),
        ('score', 'The aggregated score for this metric, subject and view'),
        ('rank_overall', 'Rank within the population schools present in this view (1 = best)'),
        ('pct_overall', 'Percentile within this view (100 = best); empty where the view ranks '
                        'fewer than 8 schools'),
        ('n_in_view', 'Number of schools present in this view (rank denominator)'),
        ('n_students', 'Median number of takers per year in this view. For composite_min: '
                       'the takers of the subject that produced the minimum.'),
        ('', ''),
        ('VIEW_KIND', 'MEANING'),
        ('base', 'Score over all the years the school has data for'),
        ('loo', "Leave-one-out: score with one of the school's own years excluded"),
        ('single_year', 'Score from one single year alone'),
        ('last_k', 'Score over the most recent k years (k = 2 .. n_years - 1)'),
        ('', ''),
        ('SUBJECT', 'SOURCE SUBJECT (CIE column, normalised)'),
        *subject_rows,
        ('composite_min', 'Minimum of the three basic subject scores for this metric/view; '
                          "identifies the school's weakest basic subject"),
        ('', ''),
        ('NOTE', 'New-formula matura (EM2023), CIE national publication. Extended subjects '
                 'describe the students who chose them, a different share of each school.'),
    ]
    return pd.DataFrame(rows, columns=['key', 'description'])


row_counts = {}
for population in POPULATIONS:
    out_dir = OUTPUT_DIR / population
    out_dir.mkdir(parents=True, exist_ok=True)
    for metric in EXPORT_METRICS:
        long_frame = build_long_frame(population, metric)
        legend = build_legend(population, metric)
        out_path = out_dir / f'schools-{metric}.xlsx'
        commit_export(
            out_path,
            unchanged=(not FORCE_REGENERATE) and xlsx_unchanged(out_path, long_frame, legend),
            write=lambda p=out_path, f=long_frame, g=legend: write_xlsx(f, g, p),
            detail=f'{len(long_frame):,} rows',
            force_regenerate=FORCE_REGENERATE,
            write_output_files=WRITE_OUTPUT_FILES,
        )
        row_counts[(population, metric)] = len(long_frame)
for population in POPULATIONS:
    counts = ', '.join(f'{m} {row_counts[(population, m)]:,}' for m in EXPORT_METRICS)
    print(f'xlsx rows {population}: {counts}')

  schools-mean.xlsx: no changes, skipping


  schools-median.xlsx: no changes, skipping


  schools-diff_mean.xlsx: no changes, skipping


  schools-unit_norm_diff_mean.xlsx: no changes, skipping


  schools-mean.xlsx: no changes, skipping


...

  schools-unit_norm_diff_mean.xlsx: no changes, skipping


  schools-mean.xlsx: no changes, skipping


  schools-median.xlsx: no changes, skipping


  schools-diff_mean.xlsx: no changes, skipping


  schools-unit_norm_diff_mean.xlsx: no changes, skipping
xlsx rows liceum: mean 215,420, median 215,420, diff_mean 215,075, unit_norm_diff_mean 215,075
xlsx rows technikum: mean 93,472, median 93,472, diff_mean 93,070, unit_norm_diff_mean 93,070
xlsx rows all: mean 251,025, median 251,025, diff_mean 250,796, unit_norm_diff_mean 250,796
